# Libera L1B - Observation Geometry Fields

The L1B radiometer product carries the observation geometry alongside the radiances: where the
boresight hit the ground, where the spacecraft and Sun were, and the angles relating them.
Downstream products depend on these - footprint matching against the WFOV camera, angular
distribution models, and flux inversion all key off them - so it is worth knowing what each field
means and what a correct one looks like.

This notebook has two halves:

1. **The product.** Open a bundled L1B granule, inventory the geometry fields, plot them, and run a
   set of internal-consistency checks that must hold if the fields agree with each other.
2. **The source.** Reproduce the same fields from SPICE with `libera_rad.geolocation`, confirm they
   match the product to float32 rounding, and compute them on a different time grid.

Each family of fields in the first half opens with a diagram of the geometry it describes - where the
spacecraft is, where it is looking, which angle is which - before any of its values are plotted. The
time series only mean something once the picture does.

Everything runs on data in this repository:

| What | Where |
| --- | --- |
| L1B granule (30 s, 100 Hz, 3000 samples) | `learning_notebooks/sample_data/LIBERA_L1B_RAD-4CH_V0-6-1_*.nc` |
| SPICE kernels it was produced from | `tests/test_data/l1b_integration_data/` (AZROT-CK, ELSCAN-CK, JPSS-CK, JPSS-SPK) |

The granule is a cross-track scan: the azimuth motor is parked and the elevation motor sweeps the
boresight through about six full cycles - twelve limb-to-limb traversals - in 30 seconds.

In [ ]:
import logging
import warnings
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr


def repo_root(start: Path | None = None) -> Path:
    """Walk up from `start` until the directory containing pyproject.toml is found."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from " + str(here))


ROOT = repo_root()
SAMPLE_DIR = ROOT / "learning_notebooks" / "sample_data"
KERNEL_DIR = ROOT / "tests" / "test_data" / "l1b_integration_data"

logging.getLogger().setLevel(logging.WARNING)

# Samples whose boresight misses the ellipsoid are expected, and curryer returns NaN for them via an
# invalid sqrt. Silencing that keeps the off-Earth samples from burying every cell in warnings.
np.seterr(invalid="ignore")

xr.set_options(display_width=120, display_max_rows=40)
np.set_printoptions(linewidth=120, suppress=True)
pd.set_option("display.width", 160, "display.max_rows", 80, "display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (12, 4.5), "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

print("repository root:", ROOT)

## 1. The product as delivered

Open the granule the way a downstream consumer would. `mask_and_scale=True` (the xarray default)
turns the product `_FillValue` entries into NaN, which is what we want for plotting and for
reasoning about coverage.

In [ ]:
l1b_path = sorted(SAMPLE_DIR.glob("LIBERA_L1B_RAD-4CH_V0-6-1_*.nc"))[-1]
ds = xr.open_dataset(l1b_path).load()

time = ds["radiometer_time"].values
seconds = (time - time[0]) / np.timedelta64(1, "s")

print("file          :", l1b_path.name)
print("dimensions    :", dict(ds.sizes))
print("time span     :", time[0], "->", time[-1])
print("cadence       : %.1f Hz over %.2f s" % (1.0 / np.median(np.diff(seconds)), seconds[-1]))
print("algorithm ver :", ds.attrs.get("algorithm_version"))
print("earth-sun AU  : %.6f" % float(ds.attrs["Earth_Sun_Distance_AU"]))
print("variables     :", len(ds.data_vars))

### 1.1 The geometry fields, grouped

Five families. The distinction that matters most when reading the product: the **surface** fields
describe the point the boresight hit and are NaN whenever it missed the Earth, whereas the
**spacecraft** fields are defined at every sample regardless of where the instrument pointed.

The table is built from the file's own attributes, so it cannot drift from the product definition.
`% finite` is the fraction of samples that are neither fill nor NaN.

In [ ]:
FIELD_GROUPS = {
    "boresight surface point": {
        "Latitude": "geodetic latitude of the boresight ellipsoid intersection (WGS84)",
        "Longitude": "geodetic longitude of the same point",
        "Colatitude": "90 - Latitude",
        "Altitude": "height of the intersection above the ellipsoid (0 by construction)",
        "Terrain_Corrected_Latitude": "not yet implemented - fill",
        "Terrain_Corrected_Longitude": "not yet implemented - fill",
        "Terrain_Corrected_Altitude": "not yet implemented - fill",
    },
    "reference ground points": {
        "Subsatellite_Latitude": "ground point directly beneath the spacecraft",
        "Subsatellite_Longitude": "ground point directly beneath the spacecraft",
        "Subsatellite_Colatitude": "90 - Subsatellite_Latitude",
        "Subsolar_Latitude": "ground point directly beneath the Sun",
        "Subsolar_Longitude": "ground point directly beneath the Sun",
        "Subsolar_Colatitude": "90 - Subsolar_Latitude",
    },
    "viewing and illumination": {
        "Viewing_Zenith_Surface": "zenith angle of the spacecraft seen from the surface point",
        "Solar_Zenith_Surface": "zenith angle of the Sun at the surface point",
        "Viewing_Azimuth_Surface_WRT_North": "azimuth to the spacecraft, clockwise from geodetic North",
        "Solar_Azimuth_Surface_WRT_North": "azimuth to the Sun, clockwise from geodetic North",
        "Relative_Azimuth_Surface": "mod(viewing - solar + 180, 360); the Sun sits at 180",
    },
    "scan and pointing": {
        "Cone_Angle": "boresight angle off the spacecraft-to-geocenter (nadir) vector",
        "Cone_Angle_Rate": "time derivative of Cone_Angle",
        "Clock_Angle": "boresight azimuth about nadir in the inertial-velocity frame (CERES SCI-12)",
        "Clock_Angle_Rate": "time derivative of Clock_Angle; filled near nadir where it is singular",
        "Along_Track_Angle": "boresight look angle from nadir in the velocity-nadir plane",
        "Cross_Track_Angle": "boresight look angle from nadir in the cross-track-nadir plane",
        "Line_Of_Sight": "boresight unit vector in J2000",
        "Azimuth": "azimuth motor encoder angle (SPICE CK Euler angle, not a curryer field)",
        "Elevation": "elevation motor encoder angle (SPICE CK Euler angle, not a curryer field)",
    },
    "spacecraft state": {
        "Satellite_Position": "spacecraft position in J2000",
        "Satellite_Velocity": "spacecraft velocity in J2000",
        "Satellite_Position_Start_Of_Hour": "position on a fixed 24-hour grid, not the sample grid",
        "Satellite_Velocity_Start_Of_Hour": "velocity on the same 24-hour grid",
        "Radius_of_Satellite_from_Center_of_Earth": "distance from the geocenter",
        "Satellite_Attitude_Q0": "body attitude quaternion, scalar part (Earth-fixed target frame)",
        "Satellite_Attitude_Q1": "body attitude quaternion, vector part",
        "Satellite_Attitude_Q2": "body attitude quaternion, vector part",
        "Satellite_Attitude_Q3": "body attitude quaternion, vector part",
    },
}


def inventory(dataset: xr.Dataset, groups: dict) -> pd.DataFrame:
    """Tabulate each grouped variable's declared metadata and finite fraction."""
    rows = []
    for group, members in groups.items():
        for name, description in members.items():
            values = dataset[name].values.astype("float64")
            attrs = dataset[name].attrs
            valid = attrs.get("valid_range")
            rows.append(
                {
                    "group": group,
                    "variable": name,
                    "dims": " x ".join(str(d) for d in dataset[name].dims),
                    "units": attrs.get("units", "-"),
                    "valid_range": f"[{valid[0]:g}, {valid[1]:g}]" if valid is not None else "-",
                    "% finite": round(100.0 * np.isfinite(values).mean(), 1),
                    "meaning": description,
                }
            )
    return pd.DataFrame(rows).set_index(["group", "variable"])


GEOMETRY_VARS = [name for members in FIELD_GROUPS.values() for name in members]
inventory(ds, FIELD_GROUPS)

Note the two `% finite` populations: the surface fields sit near 79% while everything spacecraft-
side is 100%. Section 1.5 shows that gap is entirely the Earth limb, not a kernel problem.

`Azimuth` and `Elevation` are worth calling out. They are motor encoder Euler angles read from the
CK frame chain, not curryer geometry fields, and they are relative to the motor frames rather than
to nadir or to the spacecraft attitude. In this granule the azimuth motor is parked near 360 and
the elevation motor does all the work.

### 1.2 The geometry in three dimensions

A time series of `Cone_Angle` tells you the scan is a sawtooth. It does not tell you what a cone
angle is. The diagrams below are drawn from the product's own numbers rather than sketched, so what
they show is what the file actually contains - and where a picture asserts a relation, the cell under
it prints the residual.

Drawing them needs Earth-fixed cartesian positions, and the product carries enough to rebuild them
without touching SPICE. `Subsatellite_Latitude`/`Longitude` is the point where the ellipsoid normal
through the spacecraft meets the surface, so the spacecraft sits on that normal at whatever height
reproduces `Radius_of_Satellite_from_Center_of_Earth` - a quadratic in the height, solved exactly
below. Part 1 therefore stays SPICE-free; Part 3 brings the kernels in.

In [ ]:
A_EARTH, B_EARTH = 6378.1366, 6356.7519   # Earth ellipsoid semi-axes in km, as curryer uses them
AU_KM = 149597870.7


def unit(v):
    """Normalise along the last axis."""
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


def geodetic_to_ecef(lat_deg, lon_deg, height_km=0.0):
    """Geodetic latitude, longitude and height to Earth-fixed cartesian kilometres."""
    phi, lam = np.radians(lat_deg), np.radians(lon_deg)
    e2 = 1.0 - (B_EARTH / A_EARTH) ** 2
    prime_vertical = A_EARTH / np.sqrt(1.0 - e2 * np.sin(phi) ** 2)
    return np.stack(
        [
            (prime_vertical + height_km) * np.cos(phi) * np.cos(lam),
            (prime_vertical + height_km) * np.cos(phi) * np.sin(lam),
            (prime_vertical * (1.0 - e2) + height_km) * np.sin(phi),
        ],
        axis=-1,
    )


def geodetic_normal(lat_deg, lon_deg):
    """Outward ellipsoid normal - the local vertical - at a geodetic latitude and longitude."""
    phi, lam = np.radians(lat_deg), np.radians(lon_deg)
    return np.stack([np.cos(phi) * np.cos(lam), np.cos(phi) * np.sin(lam), np.sin(phi)], axis=-1)


def spacecraft_ecef(sub_lat, sub_lon, radius):
    """Earth-fixed spacecraft position from its subsatellite point and geocentric radius."""
    foot = geodetic_to_ecef(sub_lat, sub_lon)
    normal = geodetic_normal(sub_lat, sub_lon)
    projected = (foot * normal).sum(-1)
    height = -projected + np.sqrt(projected**2 - ((foot * foot).sum(-1) - radius**2))
    return foot + height[..., None] * normal


def orbital_frame(positions, forwards):
    """Per-sample nadir, in-track and cross-track unit vectors, in whatever frame the inputs are in."""
    nadir = -unit(positions)
    in_track = unit(forwards - (forwards * nadir).sum(-1)[..., None] * nadir)
    return nadir, in_track, np.cross(nadir, in_track)


def angle_between(a, b):
    """Angle in degrees between two directions, stable when they very nearly coincide.

    `arccos` of a dot product loses half its digits near zero angle, which matters here: several of
    the numbers worth quoting are hundredths of a degree, and float32 storage alone is enough to turn
    those into tenths through `arccos`.
    """
    a, b = unit(np.asarray(a, dtype="float64")), unit(np.asarray(b, dtype="float64"))
    return np.degrees(np.arctan2(np.linalg.norm(np.cross(a, b), axis=-1), (a * b).sum(-1)))


def quaternion_matrix(scalar_first):
    """Rotation matrix from a (scalar, vector) quaternion; the columns are the rotated axes."""
    # The product stores the quaternion as float32, so renormalise before differentiating directions.
    w, x, y, z = (unit(np.asarray(scalar_first, dtype="float64"))[..., i] for i in range(4))
    return np.stack(
        [
            np.stack([1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)], axis=-1),
            np.stack([2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)], axis=-1),
            np.stack([2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)], axis=-1),
        ],
        axis=-2,
    )


def turning_points(angle):
    """Sample indices where a scan angle reverses direction, skipping fill."""
    valid = np.flatnonzero(np.isfinite(angle))
    reversal = np.flatnonzero(np.diff(np.sign(np.diff(angle[valid]))) != 0) + 1
    return valid[reversal]

In [ ]:
COASTLINES_ECEF = [
    geodetic_to_ecef(np.asarray(line.coords)[:, 1], np.asarray(line.coords)[:, 0])
    for geometry in cfeature.COASTLINE.geometries()
    for line in getattr(geometry, "geoms", [geometry])
    if len(np.asarray(line.coords)) > 1
]


def camera_angles(direction):
    """The matplotlib (elev, azim) pair that puts the camera along `direction`."""
    d = unit(np.asarray(direction, dtype="float64"))
    return float(np.degrees(np.arcsin(d[2]))), float(np.degrees(np.arctan2(d[1], d[0])))


def axes3d(fig, *subplot_spec):
    """3D axes with matplotlib's depth sorting off, so explicit zorder decides what covers what."""
    return fig.add_subplot(*subplot_spec, projection="3d", computed_zorder=False)


def frame3d(ax, camera, half_width, center=(0.0, 0.0, 0.0), zoom=1.0):
    """Aim the camera, give the three axes one equal scale, and drop the box."""
    elev, azim = camera_angles(camera)
    ax.view_init(elev=elev, azim=azim)
    for set_limit, middle in zip((ax.set_xlim, ax.set_ylim, ax.set_zlim), center):
        set_limit(middle - half_width, middle + half_width)
    ax.set_box_aspect((1, 1, 1), zoom=zoom)
    ax.set_axis_off()


def near_side(points, camera):
    """Blank points on the hemisphere facing away from the camera, so they cannot draw through the globe."""
    return np.where(((points @ unit(camera)) > 0)[:, None], points, np.nan)


def draw_globe(ax, camera, coast_lw=0.45, graticule_step=30.0):
    """Ellipsoid surface, near-side coastlines and a graticule, in Earth-fixed kilometres."""
    lon_mesh, lat_mesh = np.linspace(0.0, 2 * np.pi, 150), np.linspace(0.0, np.pi, 80)
    ax.plot_surface(
        A_EARTH * np.outer(np.cos(lon_mesh), np.sin(lat_mesh)),
        A_EARTH * np.outer(np.sin(lon_mesh), np.sin(lat_mesh)),
        B_EARTH * np.outer(np.ones_like(lon_mesh), np.cos(lat_mesh)),
        color="#dce6f0", shade=False, edgecolor="none", antialiased=False, zorder=1,
    )
    for segment in COASTLINES_ECEF:
        visible = near_side(segment * 1.003, camera)
        if np.isfinite(visible).any():
            ax.plot(visible[:, 0], visible[:, 1], visible[:, 2], color="#5b6b7c", lw=coast_lw, zorder=2)
    if graticule_step:
        for latitude in np.arange(-60.0, 61.0, graticule_step):
            ring = near_side(geodetic_to_ecef(np.full(181, latitude), np.linspace(-180, 180, 181)) * 1.001, camera)
            ax.plot(ring[:, 0], ring[:, 1], ring[:, 2], color="#9fb0c0", lw=0.3, zorder=2)
        for longitude in np.arange(-180.0, 180.0, graticule_step):
            arc = near_side(geodetic_to_ecef(np.linspace(-90, 90, 91), np.full(91, longitude)) * 1.001, camera)
            ax.plot(arc[:, 0], arc[:, 1], arc[:, 2], color="#9fb0c0", lw=0.3, zorder=2)


def arc_points(from_vec, to_vec, radius, origin=(0.0, 0.0, 0.0), n=60):
    """Points along the shorter great-circle arc between two directions, about `origin`."""
    a, b = unit(np.asarray(from_vec, dtype="float64")), unit(np.asarray(to_vec, dtype="float64"))
    perpendicular = b - (b @ a) * a
    length = np.linalg.norm(perpendicular)
    if length < 1e-12:
        return np.full((n, 3), np.nan)
    sweep = np.linspace(0.0, np.arccos(np.clip(a @ b, -1.0, 1.0)), n)
    return np.asarray(origin, dtype="float64") + radius * (
        np.cos(sweep)[:, None] * a + np.sin(sweep)[:, None] * (perpendicular / length)
    )


def arc3d(ax, origin, from_vec, to_vec, radius, **kwargs):
    """The shorter great-circle arc between two directions, drawn about `origin`."""
    points = arc_points(from_vec, to_vec, radius, origin)
    ax.plot(points[:, 0], points[:, 1], points[:, 2], **kwargs)


HALO = {"bbox": dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75)}


def label3d(ax, point, text, **kwargs):
    """Text at a 3D point, centred by default."""
    kwargs.setdefault("ha", "center")
    ax.text(*np.asarray(point, dtype="float64"), text, **kwargs)


print("coastline polylines:", len(COASTLINES_ECEF))

The establishing shot. On the left, one limb-to-limb traversal: the boresight fans out from the
spacecraft and the ground points it lands on are `Latitude` and `Longitude`, while the spacecraft's
own footprint traces `Subsatellite_Latitude`/`Longitude`. Both are ground points, and conflating
them is the most common way to misread this product - so the right panel zooms in until they are
plainly different places, with the parallel and meridian through the surface point drawn in, since
those two numbers *are* the field.

In [ ]:
lat, lon = ds["Latitude"].values.astype("float64"), ds["Longitude"].values.astype("float64")
sub_lat = ds["Subsatellite_Latitude"].values.astype("float64")
sub_lon = ds["Subsatellite_Longitude"].values.astype("float64")
radius = ds["Radius_of_Satellite_from_Center_of_Earth"].values.astype("float64")
cone = ds["Cone_Angle"].values.astype("float64")
cross_track = ds["Cross_Track_Angle"].values.astype("float64")

satellite = spacecraft_ecef(sub_lat, sub_lon, radius)
ground_step = np.gradient(satellite, axis=0)
nadir_ecef, in_track_ecef, _ = orbital_frame(satellite, ground_step)
print("rebuilt |position| against Radius: max %.1e km" % np.abs(np.linalg.norm(satellite, axis=1) - radius).max())

turns = turning_points(cross_track)
traversal = np.arange(turns[0], turns[1] + 1)
traversal = traversal[np.isfinite(lat[traversal])]
close_up = traversal[int(0.80 * len(traversal))]

fig = plt.figure(figsize=(14, 7))

wide = traversal[len(traversal) // 2]
camera = unit(-nadir_ecef[wide] * np.cos(np.radians(22)) - in_track_ecef[wide] * np.sin(np.radians(22)))
ax = axes3d(fig, 1, 2, 1)
draw_globe(ax, camera)
for order, sample in enumerate(traversal[np.linspace(0, len(traversal) - 1, 13).astype(int)]):
    ax.plot(
        *np.stack([satellite[sample], geodetic_to_ecef(lat[sample], lon[sample])], axis=1),
        color="crimson", lw=0.7, alpha=0.5, zorder=7,
        label="boresight rays, one traversal" if order == 0 else None,
    )
scan_line = near_side(geodetic_to_ecef(lat[traversal], lon[traversal]) * 1.004, camera)
ground_track = near_side(geodetic_to_ecef(sub_lat, sub_lon) * 1.004, camera)
ax.plot(scan_line[:, 0], scan_line[:, 1], scan_line[:, 2], color="tab:orange", lw=2.4, zorder=8,
        label="one scan line: Latitude, Longitude")
ax.plot(ground_track[:, 0], ground_track[:, 1], ground_track[:, 2], color="tab:blue", lw=2.6, zorder=8,
        label="30 s of Subsatellite_Lat/Lon")
subsolar = geodetic_to_ecef(ds["Subsolar_Latitude"].values[0], ds["Subsolar_Longitude"].values[0])
ax.plot(*np.stack([subsolar, subsolar * 1.22], axis=1), color="goldenrod", lw=2.0, zorder=8)
ax.scatter(*subsolar * 1.01, marker="*", s=280, color="gold", edgecolor="black", lw=0.5, zorder=9,
           label="Subsolar_Lat/Lon")
ax.scatter(*satellite[wide], color="black", s=55, zorder=10)
frame3d(ax, camera, 6650, zoom=1.45)
ax.legend(loc="upper left", fontsize=8)
ax.set_title("One cross-track traversal, %.1f s of the granule" % (seconds[turns[1]] - seconds[turns[0]]), fontsize=10)

camera = unit(-nadir_ecef[close_up] * np.cos(np.radians(72)) + in_track_ecef[close_up] * np.sin(np.radians(72)))
ax = axes3d(fig, 1, 2, 2)
draw_globe(ax, camera, coast_lw=0.7, graticule_step=10.0)
surface_point = geodetic_to_ecef(lat[close_up], lon[close_up])
subsatellite_point = geodetic_to_ecef(sub_lat[close_up], sub_lon[close_up])
parallel = near_side(geodetic_to_ecef(np.full(721, lat[close_up]), np.linspace(-180, 180, 721)) * 1.003, camera)
meridian = near_side(geodetic_to_ecef(np.linspace(-90, 90, 361), np.full(361, lon[close_up])) * 1.003, camera)
ax.plot(parallel[:, 0], parallel[:, 1], parallel[:, 2], color="darkorange", lw=1.4, zorder=5,
        label="Latitude = %.2f deg" % lat[close_up])
ax.plot(meridian[:, 0], meridian[:, 1], meridian[:, 2], color="darkorange", lw=1.4, ls="--", zorder=5,
        label="Longitude = %.2f deg" % lon[close_up])
ax.plot(*np.stack([satellite[close_up], subsatellite_point], axis=1), color="tab:blue", lw=2.2, ls="--", zorder=9,
        label="nadir, %.0f km" % (radius[close_up] - np.linalg.norm(subsatellite_point)))
ax.plot(*np.stack([satellite[close_up], surface_point], axis=1), color="crimson", lw=2.6, zorder=9,
        label="Line_Of_Sight, Cone_Angle = %.1f deg" % cone[close_up])
arc3d(ax, satellite[close_up], nadir_ecef[close_up], unit(surface_point - satellite[close_up]), 420.0,
      color="black", lw=1.3, zorder=10)
ax.scatter(*satellite[close_up], color="black", s=60, zorder=11)
ax.scatter(*subsatellite_point * 1.004, color="tab:blue", s=55, zorder=11)
ax.scatter(*surface_point * 1.004, color="crimson", s=55, zorder=11)
label3d(ax, satellite[close_up] * 1.018, "spacecraft", fontsize=9, zorder=12)
label3d(ax, subsatellite_point * 0.988, "subsatellite point", color="tab:blue", fontsize=8.5, zorder=12)
label3d(ax, surface_point * 0.986, "surface point", color="crimson", fontsize=8.5, zorder=12)
frame3d(ax, camera, 1500, center=tuple(unit(0.5 * satellite[close_up] + 0.5 * surface_point) * 6900), zoom=1.45)
ax.legend(loc="upper left", fontsize=8)
ax.set_title("The boresight point is not the subsatellite point", fontsize=10)

plt.tight_layout()
plt.show()

separation = angle_between(geodetic_to_ecef(lat, lon), geodetic_to_ecef(sub_lat, sub_lon))
print("boresight to subsatellite ground separation: %.2f .. %.2f deg (%.0f .. %.0f km)"
      % (np.nanmin(separation), np.nanmax(separation),
         np.nanmin(separation) * 111.2, np.nanmax(separation) * 111.2))
print("geocentric nadir against geodetic nadir    : %.4f deg (mean)"
      % angle_between(satellite, geodetic_normal(sub_lat, sub_lon)).mean())

### 1.3 Where the instrument looked

The boresight ground point against the subsatellite track. Over 30 seconds the subsatellite point
advances under 2 degrees of latitude, while the scan drags the boresight across more than 60 degrees
of longitude - the scan dominates the picture entirely, which is exactly what a cross-track granule
should look like. The printed extents below quantify both.

In [ ]:
def globe_axes(center_lon: float, center_lat: float, figsize=(9, 9)):
    """Orthographic GeoAxes centred on the given point, with coastlines and a graticule."""
    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=ccrs.Orthographic(center_lon, center_lat))
    ax.set_global()
    ax.coastlines(linewidth=0.6)
    ax.gridlines(linewidth=0.3, linestyle="--", alpha=0.6)
    return fig, ax


lat, lon = ds["Latitude"].values, ds["Longitude"].values
sub_lat, sub_lon = ds["Subsatellite_Latitude"].values, ds["Subsatellite_Longitude"].values
on_earth = np.isfinite(lat)

fig, ax = globe_axes(float(np.nanmean(lon)), float(np.nanmean(lat)))
scatter = ax.scatter(
    lon[on_earth], lat[on_earth], c=seconds[on_earth], s=3, cmap="plasma",
    transform=ccrs.PlateCarree(), zorder=6, label="boresight ground point",
)
ax.plot(sub_lon, sub_lat, color="white", linewidth=2.5, transform=ccrs.PlateCarree(), zorder=8)
ax.plot(sub_lon, sub_lat, color="black", linewidth=1.2, transform=ccrs.PlateCarree(), zorder=9,
        label="subsatellite track")
ax.scatter(ds["Subsolar_Longitude"].values[0], ds["Subsolar_Latitude"].values[0], marker="*",
           s=320, color="gold", edgecolor="black", linewidth=0.6,
           transform=ccrs.PlateCarree(), zorder=10, label="subsolar point")
plt.colorbar(scatter, ax=ax, shrink=0.55, pad=0.03, label="seconds from granule start")
ax.legend(loc="upper left")
ax.set_title("Libera radiometer - boresight ground point and subsatellite track")
plt.show()

print("boresight  lat %.2f .. %.2f   lon %.2f .. %.2f" % (np.nanmin(lat), np.nanmax(lat), np.nanmin(lon), np.nanmax(lon)))
print("subsatellite  lat %.2f .. %.2f   lon %.2f .. %.2f" % (sub_lat.min(), sub_lat.max(), sub_lon.min(), sub_lon.max()))
print("off-Earth samples: %d of %d (%.1f%%)" % ((~on_earth).sum(), len(lat), 100 * (~on_earth).mean()))

### 1.4 The scan

`Cross_Track_Angle` and `Elevation` trace the same motion in different frames: the first is a look
angle from nadir, the second is the raw motor encoder. They are **sign-inverted** with respect to
each other - the two frames wind in opposite directions - so the panel below shows mirrored
sawtooths rather than overlapping ones. Section 2 checks that relation quantitatively; it is the one
place where motor telemetry and derived pointing can be compared directly.

`Along_Track_Angle` stays within about a degree of zero, which is the signature of a pure cross-track
scan. It is not identically zero, and it oscillates at twice the scan frequency: the fixed
boresight-to-motor-frame offset in the FK projects differently as the boresight swings, so the
residual along-track component peaks twice per sweep.

The picture first. The left panel puts the origin at the spacecraft and draws the orbital frame the
two look angles are measured in: nadir, in-track (along the inertial velocity) and cross-track. Every
boresight in one traversal is drawn, and they all lie in the cross-track/nadir plane - which is the
geometric content of "cross-track scan". The right panel is that fan seen in look-angle coordinates,
with the along-track axis stretched about sixty-fold so the residual is visible at all.

In [ ]:
position_j2000 = ds["Satellite_Position"].values.astype("float64")
velocity_j2000 = ds["Satellite_Velocity"].values.astype("float64")
line_of_sight = ds["Line_Of_Sight"].values.astype("float64")
along_track = ds["Along_Track_Angle"].values.astype("float64")

sweep = np.arange(turns[0], turns[1] + 1)
next_sweep = np.arange(turns[1], turns[2] + 1)
middle = sweep[len(sweep) // 2]
marked = sweep[int(0.78 * len(sweep))]
nadir, in_track, cross_axis = orbital_frame(position_j2000, velocity_j2000)

fig = plt.figure(figsize=(14, 6.2))

ax = axes3d(fig, 1, 2, 1)
span, height = np.meshgrid(np.linspace(-1.15, 1.15, 2), np.linspace(-0.05, 1.10, 2))
plane = span[..., None] * cross_axis[middle] + height[..., None] * nadir[middle]
ax.plot_surface(plane[..., 0], plane[..., 1], plane[..., 2], color="tab:purple", alpha=0.12,
                shade=False, edgecolor="none", zorder=1)
for order, sample in enumerate(sweep[np.linspace(0, len(sweep) - 1, 25).astype(int)]):
    ax.plot(*np.stack([np.zeros(3), line_of_sight[sample]], axis=1), color="crimson", lw=0.8, alpha=0.55,
            zorder=6, label="boresight, one traversal" if order == 0 else None)
for vector, colour, name in ((nadir[middle], "tab:blue", "nadir"), (in_track[middle], "tab:green", "in-track"),
                             (cross_axis[middle], "tab:purple", "cross-track")):
    ax.plot(*np.stack([np.zeros(3), vector], axis=1), color=colour, lw=2.4, zorder=8)
    label3d(ax, vector * 1.26, name, color=colour, fontsize=9, zorder=11, **HALO)
arc3d(ax, np.zeros(3), nadir[marked], line_of_sight[marked], 0.55, color="black", lw=1.5, zorder=9)
label3d(ax, unit(nadir[marked] + line_of_sight[marked]) * 0.66, "Cone_Angle", fontsize=9, zorder=11, **HALO)
ax.scatter(0, 0, 0, color="black", s=60, zorder=12)
frame3d(ax, unit(0.85 * in_track[middle] - 0.42 * nadir[middle] + 0.30 * cross_axis[middle]), 1.12, zoom=1.4)
ax.legend(loc="upper left", fontsize=8.5)
ax.set_title("The scan stays in the cross-track / nadir plane", fontsize=10)

ax = fig.add_subplot(1, 2, 2)
for indices, style in ((sweep, dict(lw=3.0, color="tab:blue", label="traversal 1")),
                      (next_sweep, dict(lw=1.3, color="tab:orange", ls="--", label="traversal 2, on top of it"))):
    ax.plot(along_track[indices], cross_track[indices], **style)
for level in (20.0, 40.0, 60.0):
    ax.axhline(level, color="grey", lw=0.7, ls=":")
    ax.axhline(-level, color="grey", lw=0.7, ls=":")
    ax.annotate("Cone_Angle = %.0f deg" % level, (-1.15, level), fontsize=8, color="grey", va="bottom")
ax.axhline(0.0, color="black", lw=0.6)
ax.axvline(0.0, color="black", lw=0.6)
ax.scatter([along_track[marked]], [cross_track[marked]], s=60, facecolor="none", edgecolor="crimson", lw=1.8,
           zorder=6, label="the marked sample")
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-80, 80)
ax.set_xlabel("Along_Track_Angle (deg) - note the axis scale")
ax.set_ylabel("Cross_Track_Angle (deg)")
ax.set_title("Two traversals in look-angle coordinates", fontsize=10)
ax.legend(loc="lower right", fontsize=8.5)

plt.tight_layout()
plt.show()

nadir_component = (line_of_sight * nadir).sum(1)
print("tan(Along_Track_Angle) vs (LOS . in-track)/(LOS . nadir) : max %.2e"
      % np.nanmax(np.abs(np.tan(np.radians(along_track)) - (line_of_sight * in_track).sum(1) / nadir_component)))
print("tan(Cross_Track_Angle) vs (LOS . cross)/(LOS . nadir)    : max %.2e"
      % np.nanmax(np.abs(np.tan(np.radians(cross_track)) - (line_of_sight * cross_axis).sum(1) / nadir_component)))
print("so the two look angles are the boresight's tangent-plane coordinates about nadir,")
print("and Along_Track_Angle stays inside %.2f deg of zero for the whole granule" % np.nanmax(np.abs(along_track)))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(seconds, ds["Cross_Track_Angle"].values, lw=0.9, label="Cross_Track_Angle")
axes[0].plot(seconds, ds["Elevation"].values, lw=0.9, ls="--", label="Elevation (motor encoder)")
axes[0].set_ylabel("degrees")
axes[0].set_title("Cross-track scan: look angle from nadir vs motor encoder")

axes[1].plot(seconds, ds["Cone_Angle"].values, lw=0.9, color="tab:green", label="Cone_Angle")
axes[1].fill_between(seconds, 0, 90, where=~on_earth, color="grey", alpha=0.25, label="boresight off Earth")
axes[1].set_ylabel("degrees")
axes[1].set_ylim(0, 90)
axes[1].set_title("Cone angle (off-nadir); shading marks samples that miss the ellipsoid")

axes[2].plot(seconds, ds["Along_Track_Angle"].values, lw=0.9, color="tab:red", label="Along_Track_Angle")
axes[2].set_ylabel("degrees")
axes[2].set_xlabel("seconds from granule start")
axes[2].set_title("Along-track angle stays near zero in a cross-track scan")

for ax in axes:
    ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

scan = ds["Cross_Track_Angle"].values
crossings = np.where(np.diff(np.signbit(np.nan_to_num(scan))))[0]
print("nadir crossings in 30 s: %d  (~%.1f s per half sweep)" % (len(crossings), seconds[-1] / max(len(crossings), 1)))
print("Along_Track_Angle range: %.3f .. %.3f deg" % (np.nanmin(ds["Along_Track_Angle"]), np.nanmax(ds["Along_Track_Angle"])))
print("Azimuth range          : %.4f .. %.4f deg" % (np.nanmin(ds["Azimuth"]), np.nanmax(ds["Azimuth"])))

### 1.5 The Earth limb decides which samples are valid

Every surface field shares one NaN mask, and that mask is set by a single geometric condition: the
boresight cone angle exceeding the limb half-angle `asin(R_earth / R_spacecraft)`. If the masks ever
disagree, or the cut-off is not at the limb, something is wrong upstream - a kernel gap would show
up as a contiguous block in time instead.

Drawn to scale, that condition is almost trivially geometric. The left panel is the true proportion
of the problem - the spacecraft sits only 13% of an Earth radius above the surface - and the right
panel zooms into the fan. Rays inside the limb half-angle strike the ellipsoid and carry surface
values; rays outside it graze past into space and every surface field is fill.

In [ ]:
cross_section = (turns[0] + turns[1]) // 2
orbit_radius = radius[cross_section]

bearing = np.linspace(-np.pi, np.pi, 1441)
directions = np.cos(bearing)[:, None] * (-nadir[cross_section]) + np.sin(bearing)[:, None] * cross_axis[cross_section]
ellipse_radius = 1.0 / np.sqrt(
    (directions[:, 0] ** 2 + directions[:, 1] ** 2) / A_EARTH**2 + directions[:, 2] ** 2 / B_EARTH**2
)
ellipse_x, ellipse_y = ellipse_radius * np.sin(bearing), ellipse_radius * np.cos(bearing)
surface_radius = float(np.interp(0.0, bearing, ellipse_radius))
limb = np.degrees(np.arcsin(surface_radius / orbit_radius))


def ray_end(cone_deg, sign=1.0, miss_length=4200.0):
    """Where a ray at this cone angle stops: the ellipsoid if it reaches it, otherwise a fixed length."""
    direction = np.array([sign * np.sin(np.radians(cone_deg)), -np.cos(np.radians(cone_deg))])
    origin = np.array([0.0, orbit_radius])
    projected = orbit_radius * direction[1]
    discriminant = projected**2 - (orbit_radius**2 - surface_radius**2)
    distance = -projected - np.sqrt(discriminant) if discriminant >= 0 else miss_length
    return origin + distance * direction


fig = plt.figure(figsize=(13.5, 5.6))
grid = fig.add_gridspec(1, 2, width_ratios=(1.0, 2.3), wspace=0.16)

for panel in range(2):
    ax = fig.add_subplot(grid[0, panel])
    zoomed = panel == 1
    ax.fill(ellipse_x, ellipse_y, color="#dce6f0", zorder=2)
    ax.plot(ellipse_x, ellipse_y, color="#5b6b7c", lw=1.1, zorder=3)
    for sign in (1.0, -1.0):
        tip = ray_end(limb, sign, miss_length=4200.0)
        ax.plot([0, tip[0]], [orbit_radius, tip[1]], color="dimgrey", lw=1.5, ls=(0, (5, 3)), zorder=6,
                label="limb ray" if (zoomed and sign > 0) else None)
        graze = ray_end(limb - 1e-9, sign)
        ax.plot([graze[0]], [graze[1]], marker="o", ms=5, color="dimgrey", zorder=8)
    for level in np.arange(0.0, 91.0, 10.0):
        reaches = level < limb
        for sign in (1.0, -1.0):
            end = ray_end(level, sign, miss_length=4200.0)
            ax.plot([0, end[0]], [orbit_radius, end[1]], color="crimson" if reaches else "grey",
                    lw=1.1 if reaches else 0.8, ls="-" if reaches else (0, (2, 3)), zorder=7)
            if reaches:
                ax.plot([end[0]], [end[1]], marker="o", ms=4.5, color="crimson", zorder=8)
        if zoomed and level in (20.0, 40.0, 60.0, 80.0):
            # Labels sit well out along each ray so they spread by angle instead of bunching up at the
            # spacecraft, but never past the ray's own end.
            reach = min(1450.0, 0.85 * np.linalg.norm(ray_end(level, 1.0, 1450.0) - np.array([0.0, orbit_radius])))
            at = np.array([0.0, orbit_radius]) + reach * np.array([np.sin(np.radians(level)),
                                                                   -np.cos(np.radians(level))])
            ax.annotate("%.0f" % level, at, fontsize=8.5, ha="center", va="center", zorder=9,
                        color="crimson" if reaches else "grey",
                        bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.85))
    ax.plot([0, 0], [orbit_radius, surface_radius], color="tab:blue", lw=2.0, ls="--", zorder=8)
    ax.plot([0], [orbit_radius], marker="o", color="black", ms=8, zorder=10)
    ax.set_aspect("equal")
    ax.grid(alpha=0.2)
    if zoomed:
        opening = np.linspace(0.0, np.radians(limb), 80)
        ax.plot(0.055 * orbit_radius * np.sin(opening), orbit_radius * (1 - 0.055 * np.cos(opening)),
                color="black", lw=1.3, zorder=9)
        ax.annotate("limb half-angle = asin(R_surface / R) = %.2f deg" % limb, (0.02, 0.07),
                    xycoords="axes fraction", fontsize=9, zorder=10,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85))
        ax.annotate("spacecraft", (0, orbit_radius), textcoords="offset points", xytext=(0, 10),
                    fontsize=9.5, ha="center")
        ax.annotate("0", (0.0, orbit_radius - 0.45 * (orbit_radius - surface_radius)),
                    textcoords="offset points", xytext=(-13, 0), fontsize=8.5, color="crimson",
                    ha="center", va="center", zorder=9)
        ax.set_xlim(-3900, 3900)
        ax.set_ylim(5460, 7480)
        ax.set_xlabel("cross-track offset from the geocenter (km)")
        ax.set_title("Cone_Angle in degrees; solid rays reach the ellipsoid, dotted rays miss it", fontsize=10)
        ax.legend(loc="lower right", fontsize=8.5)
    else:
        ax.set_xlim(-1.12 * A_EARTH, 1.12 * A_EARTH)
        ax.set_ylim(-1.12 * A_EARTH, 1.20 * A_EARTH)
        ax.set_xlabel("cross-track offset (km)")
        ax.set_ylabel("distance along the nadir axis (km)")
        ax.set_title("To scale: %.0f km up over a %.0f km Earth" % (orbit_radius - surface_radius, surface_radius),
                     fontsize=10)

plt.show()

on_earth = np.isfinite(lat)
print("limb half-angle under the spacecraft : %.3f deg" % limb)
print("largest Cone_Angle still on Earth    : %.3f deg" % np.nanmax(cone[on_earth]))
print("smallest Cone_Angle off Earth        : %.3f deg" % np.nanmin(cone[~on_earth]))
print("the cut-off sits between the limb computed under the spacecraft and the one at the grazing point,")
print("which lies at a different latitude and so a slightly different ellipsoid radius")

In [ ]:
SURFACE_FIELDS = [
    "Latitude", "Longitude", "Colatitude", "Altitude",
    "Viewing_Zenith_Surface", "Solar_Zenith_Surface",
    "Viewing_Azimuth_Surface_WRT_North", "Solar_Azimuth_Surface_WRT_North",
    "Relative_Azimuth_Surface",
]

reference_mask = ~np.isfinite(ds["Latitude"].values)
mask_report = pd.DataFrame(
    [
        {
            "variable": name,
            "n_nan": int(np.isnan(ds[name].values).sum()),
            "mask identical to Latitude": bool(np.array_equal(np.isnan(ds[name].values), reference_mask)),
        }
        for name in SURFACE_FIELDS
    ]
).set_index("variable")
print(mask_report.to_string(), "\n")

cone = ds["Cone_Angle"].values.astype("float64")
radius = ds["Radius_of_Satellite_from_Center_of_Earth"].values.astype("float64")
limb_angle = np.degrees(np.arcsin(6378.1366 / radius.mean()))

print("largest cone angle still on Earth : %.3f deg" % np.nanmax(cone[~reference_mask]))
print("smallest cone angle off Earth     : %.3f deg" % np.nanmin(cone[reference_mask]))
print("geometric limb asin(Re/Rsc)       : %.3f deg" % limb_angle)
print("off-Earth samples contiguous?     :", np.diff(np.where(reference_mask)[0]).max() > 1 and "no - several sweeps" or "yes")

### 1.6 Viewing and illumination angles

The four surface angles plus their derived relative azimuth. `Viewing_Zenith_Surface` tracks the
scan almost symmetrically about nadir; `Solar_Zenith_Surface` varies much less because the Sun
barely moves in 30 seconds - its variation is dominated by the surface point moving, not by the Sun.
The bottom-right panel is the diagnostic view: viewing zenith against cross-track angle should be a
clean, near-symmetric V with no hysteresis between sweeps.

All four are angles in the local horizon frame at the surface point, so that is where the left panel
stands: local zenith up, geodetic North and East in the horizon plane, and one vector each to the
spacecraft and to the Sun. Zenith angles are measured down from the vertical, azimuths are measured
in the horizon plane clockwise from North, and the relative azimuth is the gap between the two,
offset so that 180 puts the spacecraft in the same azimuthal direction as the Sun and 0 puts it
opposite.

The right panel puts every sample in that same space at once. A cross-track scan sweeps an arc from
one horizon through nadir to the other, and where that arc passes relative to the Sun is what
`Relative_Azimuth_Surface` reports.

In [ ]:
view_zenith = ds["Viewing_Zenith_Surface"].values.astype("float64")
sun_zenith = ds["Solar_Zenith_Surface"].values.astype("float64")
view_azimuth = ds["Viewing_Azimuth_Surface_WRT_North"].values.astype("float64")
sun_azimuth = ds["Solar_Azimuth_Surface_WRT_North"].values.astype("float64")
relative_azimuth = ds["Relative_Azimuth_Surface"].values.astype("float64")
subsolar_lat = ds["Subsolar_Latitude"].values.astype("float64")
subsolar_lon = ds["Subsolar_Longitude"].values.astype("float64")

mid_scan = np.flatnonzero(np.isfinite(lat) & (np.abs(cone - 45.0) < 1.5))
marker = int(mid_scan[len(mid_scan) // 2])

surface = geodetic_to_ecef(lat[marker], lon[marker])
vertical = geodetic_normal(lat[marker], lon[marker])
east_axis = unit(np.cross(np.array([0.0, 0.0, 1.0]), vertical))
north_axis = np.cross(vertical, east_axis)
sun_position = geodetic_normal(subsolar_lat[marker], subsolar_lon[marker]) * float(
    ds.attrs["Earth_Sun_Distance_AU"]
) * AU_KM


def horizon_frame(target):
    """A direction from the surface point expressed as (east, north, up) components."""
    d = unit(target - surface)
    return np.array([d @ east_axis, d @ north_axis, d @ vertical])


to_spacecraft, to_sun = horizon_frame(satellite[marker]), horizon_frame(sun_position)
UP, NORTH, EAST = np.eye(3)[[2, 1, 0]]

fig = plt.figure(figsize=(14, 6.6))

ax = axes3d(fig, 1, 2, 1)
disc_radius, disc_angle = np.meshgrid(np.linspace(0.0, 1.05, 2), np.linspace(0.0, 2 * np.pi, 80))
ax.plot_surface(disc_radius * np.cos(disc_angle), disc_radius * np.sin(disc_angle), np.zeros_like(disc_radius),
                color="#cfd9e4", alpha=0.45, shade=False, edgecolor="none", zorder=1)
horizon = np.linspace(0.0, 2 * np.pi, 240)
ax.plot(np.cos(horizon), np.sin(horizon), np.zeros_like(horizon), color="#7d8a99", lw=0.8, zorder=2)
for vector, colour, name in ((UP, "black", "zenith"), (NORTH, "tab:green", "North"), (EAST, "tab:red", "East")):
    ax.plot(*np.stack([np.zeros(3), vector], axis=1), color=colour, lw=2.0, zorder=8)
    label3d(ax, vector * 1.12, name, color=colour, fontsize=9.5, zorder=11)
for vector, colour, name in ((to_spacecraft, "tab:blue", "to spacecraft"), (to_sun, "goldenrod", "to Sun")):
    flat = np.array([vector[0], vector[1], 0.0])
    ax.plot(*np.stack([np.zeros(3), vector], axis=1), color=colour, lw=2.6, zorder=9)
    ax.plot(*np.stack([vector, flat], axis=1), color=colour, lw=0.9, ls=":", zorder=7)
    ax.plot(*np.stack([np.zeros(3), flat], axis=1), color=colour, lw=1.2, ls="--", zorder=7)
    label3d(ax, vector * 1.14, name, color=colour, fontsize=9, zorder=11)
flat_spacecraft = unit(np.array([to_spacecraft[0], to_spacecraft[1], 0.0]))
flat_sun = unit(np.array([to_sun[0], to_sun[1], 0.0]))
for start, end, radius_frac, colour, tag, dashes in (
    (UP, to_spacecraft, 0.40, "tab:blue", "VZA", "-"),
    (UP, to_sun, 0.72, "goldenrod", "SZA", "-"),
    (NORTH, flat_spacecraft, 0.34, "tab:blue", "VAA", "--"),
    (NORTH, flat_sun, 0.60, "goldenrod", "SAA", "--"),
    (flat_spacecraft, flat_sun, 0.86, "tab:purple", "RAA", "-"),
):
    arc3d(ax, np.zeros(3), start, end, radius_frac, color=colour, lw=1.5, ls=dashes, zorder=9)
    label3d(ax, unit(start + end) * (radius_frac + 0.07), tag, color=colour, fontsize=9, zorder=12)
ax.scatter(0, 0, 0, color="crimson", s=55, zorder=12)
label3d(ax, np.array([0.0, 0.0, -0.12]), "surface point", color="crimson", fontsize=9, zorder=12)
ax.text2D(
    0.0, 0.0,
    "VZA  Viewing_Zenith_Surface              %6.2f deg\n"
    "SZA  Solar_Zenith_Surface                %6.2f deg\n"
    "VAA  Viewing_Azimuth_Surface_WRT_North   %6.2f deg\n"
    "SAA  Solar_Azimuth_Surface_WRT_North     %6.2f deg\n"
    "RAA  Relative_Azimuth_Surface            %6.2f deg"
    % (view_zenith[marker], sun_zenith[marker], view_azimuth[marker], sun_azimuth[marker],
       relative_azimuth[marker]),
    transform=ax.transAxes, fontsize=8, family="monospace", va="bottom",
)
frame3d(ax, unit(0.55 * EAST - 0.75 * NORTH + 0.42 * UP), 1.15, zoom=1.42)
ax.set_title("The four surface angles in the local horizon frame", fontsize=10)

ax = fig.add_subplot(1, 2, 2, projection="polar")
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
finite = np.isfinite(view_zenith)
scatter = ax.scatter(np.radians(view_azimuth[finite]), view_zenith[finite], c=seconds[finite], s=4,
                     cmap="plasma", zorder=5)
principal = np.radians(sun_azimuth[marker])
ax.plot([principal, principal + np.pi], [90, 90], color="goldenrod", lw=1.2, ls="--", zorder=4,
        label="solar principal plane")
ax.scatter(np.radians(sun_azimuth[marker]), sun_zenith[marker], marker="*", s=380, color="gold",
           edgecolor="black", lw=0.6, zorder=8)
ax.annotate("Sun", (np.radians(sun_azimuth[marker]), sun_zenith[marker]), textcoords="offset points",
            xytext=(12, 6), fontsize=9)
ax.scatter(np.radians(view_azimuth[marker]), view_zenith[marker], s=80, facecolor="none",
           edgecolor="tab:blue", lw=1.8, zorder=9, label="the marked sample")
ax.set_rlim(0, 90)
ax.set_rlabel_position(130)
ax.set_title("Every sample in view geometry:\nradius is zenith angle, angle is azimuth from North", fontsize=10)
ax.legend(loc="lower left", bbox_to_anchor=(-0.14, -0.09), fontsize=8.5)
plt.colorbar(scatter, ax=ax, shrink=0.7, pad=0.10, label="seconds from granule start")

plt.tight_layout()
plt.show()

# Rebuild all of them, each sample in its own horizon frame rather than the marked sample's.
surface_all = geodetic_to_ecef(lat, lon)
vertical_all = geodetic_normal(lat, lon)
east_all = unit(np.cross(np.broadcast_to(np.array([0.0, 0.0, 1.0]), vertical_all.shape), vertical_all))
north_all = np.cross(vertical_all, east_all)
to_satellite = unit(satellite - surface_all)
print("rebuilt from the local horizon frame, over all %d on-Earth samples:" % int(np.isfinite(lat).sum()))
print("  Viewing_Zenith  max |diff| %.5f deg"
      % np.nanmax(np.abs(angle_between(to_satellite, vertical_all) - view_zenith)))
print("  Viewing_Azimuth max |diff| %.5f deg"
      % np.nanmax(np.abs(np.degrees(np.arctan2((to_satellite * east_all).sum(1),
                                               (to_satellite * north_all).sum(1))) % 360.0 - view_azimuth)))
print("swapping the ellipsoid normal for the geocentric radial as the vertical would move")
print("  Viewing_Zenith by up to %.4f deg, so the geodetic normal is the right vertical"
      % np.nanmax(np.abs(angle_between(to_satellite, surface_all) - view_zenith)))

In [ ]:
vza = ds["Viewing_Zenith_Surface"].values
sza = ds["Solar_Zenith_Surface"].values
vaz = ds["Viewing_Azimuth_Surface_WRT_North"].values
saz = ds["Solar_Azimuth_Surface_WRT_North"].values
raa = ds["Relative_Azimuth_Surface"].values

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0, 0].plot(seconds, vza, lw=0.9, label="Viewing_Zenith_Surface")
axes[0, 0].plot(seconds, sza, lw=0.9, label="Solar_Zenith_Surface")
axes[0, 0].set_title("Zenith angles")
axes[0, 0].set_ylabel("degrees")

axes[0, 1].plot(seconds, vaz, lw=0.9, label="Viewing_Azimuth (WRT North)")
axes[0, 1].plot(seconds, saz, lw=0.9, label="Solar_Azimuth (WRT North)")
axes[0, 1].set_title("Azimuth angles, clockwise from North")
axes[0, 1].set_ylabel("degrees")

axes[1, 0].plot(seconds, raa, lw=0.9, color="tab:purple", label="Relative_Azimuth_Surface")
axes[1, 0].axhline(180, color="gold", lw=1.5, ls="--", label="Sun direction (180)")
axes[1, 0].set_title("Relative azimuth; 180 is the solar principal plane")
axes[1, 0].set_xlabel("seconds from granule start")
axes[1, 0].set_ylabel("degrees")

axes[1, 1].scatter(ds["Cross_Track_Angle"].values, vza, c=seconds, s=3, cmap="plasma")
axes[1, 1].set_title("Viewing zenith vs cross-track angle")
axes[1, 1].set_xlabel("Cross_Track_Angle (deg)")
axes[1, 1].set_ylabel("Viewing_Zenith_Surface (deg)")

for ax in axes.flat[:3]:
    ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()

summary = pd.DataFrame(
    {"min": [np.nanmin(v) for v in (vza, sza, vaz, saz, raa)],
     "max": [np.nanmax(v) for v in (vza, sza, vaz, saz, raa)],
     "range": [np.nanmax(v) - np.nanmin(v) for v in (vza, sza, vaz, saz, raa)]},
    index=["Viewing_Zenith", "Solar_Zenith", "Viewing_Azimuth", "Solar_Azimuth", "Relative_Azimuth"],
)
print(summary.round(3).to_string())

### 1.7 Cone and clock angles

`Cone_Angle` and `Clock_Angle` are the CERES SCI-12 pair: cone is the off-nadir angle, clock is the
azimuth about nadir measured in the inertial-velocity orbital frame. A cross-track scan therefore
appears in polar coordinates as two radial spokes near clock = 90 and clock = 270, with the
boresight running out to the limb and back along each.

`Clock_Angle_Rate` is the one field the product deliberately blanks. Clock angle is an azimuth about
nadir, so its derivative is singular there: as the scan crosses nadir the azimuth flips by roughly
180 degrees faster than 100 Hz sampling can resolve, making the finite difference an aliasing
artifact rather than a derivative. `libera_rad` fills the rate inside a nadir cone set by
`CLOCK_RATE_MIN_CONE_ANGLE_DEG`, which is a product decision rather than a curryer one.

Sizing that cone is not arbitrary. The largest surviving rate always sits at the gate boundary and
falls off as one over the gate squared: for a scan of angular speed `w` whose closest approach to
nadir is `b`, the rate at cone angle `g` is `w*b/g**2`. The gate is therefore chosen to bring the
surviving rate inside the field's declared `valid_range` with margin - the dotted lines in the right
panel mark that range.

Cone and clock are just polar coordinates for the boresight, centred on nadir, and the left panel
draws them that way: the cone opens around the nadir axis, its rim is the locus of one cone angle,
and clock is the position along that rim - zero at the in-track direction, increasing toward
`nadir x in-track`, which is to the right of the velocity vector.

The middle and right panels are the same crossing seen two ways, and together they are the whole
argument for gating the rate. In look-angle coordinates the boresight passes within half a degree of
the origin; clock is the azimuth of that point, so it swings almost 180 degrees while the boresight
moves a fraction of a degree, and the finite difference behind `Clock_Angle_Rate` blows up. The gate
is the red circle: inside it the rate is fill.

In [ ]:
from libera_rad.constants import CLOCK_RATE_MIN_CONE_ANGLE_DEG

clock_deg = ds["Clock_Angle"].values.astype("float64")
gate = float(CLOCK_RATE_MIN_CONE_ANGLE_DEG)

shown = np.flatnonzero(np.isfinite(cone) & (np.abs(cone - 40.0) < 1.0))
marked = int(shown[len(shown) // 2])
components = np.array([line_of_sight[marked] @ in_track[marked], line_of_sight[marked] @ cross_axis[marked],
                       -line_of_sight[marked] @ nadir[marked]])
boresight = unit(components)
opening = np.radians(cone[marked])
IN_TRACK, RIGHT, ZENITH = np.eye(3)

fig = plt.figure(figsize=(16, 5.4))
grid = fig.add_gridspec(1, 3, width_ratios=(1.4, 1.0, 1.1), wspace=0.30)

ax = axes3d(fig, grid[0, 0])
disc_radius, disc_angle = np.meshgrid(np.linspace(0.0, 0.95, 2), np.linspace(0.0, 2 * np.pi, 80))
ax.plot_surface(disc_radius * np.cos(disc_angle), disc_radius * np.sin(disc_angle), np.zeros_like(disc_radius),
                color="#cfd9e4", alpha=0.40, shade=False, edgecolor="none", zorder=1)
azimuth = np.linspace(0.0, 2 * np.pi, 200)
cone_reach, cone_azimuth = np.meshgrid(np.linspace(0.0, 1.0, 2), azimuth)
cone_surface = np.stack([cone_reach * np.sin(opening) * np.cos(cone_azimuth),
                         cone_reach * np.sin(opening) * np.sin(cone_azimuth),
                         -cone_reach * np.cos(opening)], axis=-1)
ax.plot_surface(cone_surface[..., 0], cone_surface[..., 1], cone_surface[..., 2], color="crimson",
                alpha=0.16, shade=False, edgecolor="none", zorder=3)
rim = np.stack([np.sin(opening) * np.cos(azimuth), np.sin(opening) * np.sin(azimuth),
                -np.cos(opening) * np.ones_like(azimuth)], axis=1)
ax.plot(rim[:, 0], rim[:, 1], rim[:, 2], color="crimson", lw=1.0, ls=":", zorder=5,
        label="constant Cone_Angle = %.0f deg" % cone[marked])
for vector, colour, name in ((-ZENITH, "tab:blue", "nadir"), (IN_TRACK, "tab:green", "in-track\nclock = 0"),
                             (RIGHT, "tab:purple", "nadir x in-track\nclock = 90")):
    ax.plot(*np.stack([np.zeros(3), vector], axis=1), color=colour, lw=2.2, zorder=8)
    label3d(ax, vector * 1.18, name, color=colour, fontsize=8.5, zorder=11, **HALO)
ax.plot(*np.stack([np.zeros(3), boresight], axis=1), color="crimson", lw=2.8, zorder=9)
label3d(ax, boresight * 1.15, "Line_Of_Sight", color="crimson", fontsize=9, zorder=11, **HALO)
flattened = unit(np.array([boresight[0], boresight[1], 0.0]))
ax.plot(*np.stack([boresight, np.array([boresight[0], boresight[1], 0.0])], axis=1), color="grey",
        lw=0.9, ls=":", zorder=7)
arc3d(ax, np.zeros(3), -ZENITH, boresight, 0.44, color="black", lw=1.5, zorder=10)
label3d(ax, unit(-ZENITH + boresight) * 0.60 - 0.12 * IN_TRACK, "cone %.0f deg" % cone[marked],
        fontsize=8.5, zorder=12, **HALO)
arc3d(ax, np.zeros(3), IN_TRACK, flattened, 0.72, color="tab:purple", lw=1.6, zorder=10)
label3d(ax, unit(IN_TRACK + flattened) * 0.86 + 0.17 * ZENITH, "clock %.0f deg" % clock_deg[marked],
        color="tab:purple", fontsize=8.5, zorder=12, **HALO)
ax.scatter(0, 0, 0, color="black", s=55, zorder=12)
frame3d(ax, unit(0.85 * IN_TRACK + 0.55 * RIGHT + 0.40 * ZENITH), 1.18, zoom=1.45)
ax.legend(loc="lower left", fontsize=8)
ax.set_title("Cone is the polar angle off nadir,\nclock the azimuth about it", fontsize=10)

ax = fig.add_subplot(grid[0, 1])
near_nadir = np.flatnonzero(np.isfinite(cone) & (cone < 20.0))
colours = ax.scatter(along_track[near_nadir], cross_track[near_nadir], c=clock_deg[near_nadir], s=9,
                     cmap="twilight", vmin=0, vmax=360, zorder=5)
ring = np.linspace(0.0, 2 * np.pi, 361)
ax.plot(np.degrees(np.arctan(np.tan(np.radians(gate)) * np.cos(ring))),
        np.degrees(np.arctan(np.tan(np.radians(gate)) * np.sin(ring))),
        color="crimson", lw=1.8, zorder=6, label="gate: Cone_Angle = %.0f deg" % gate)
ax.plot([0], [0], marker="+", ms=15, mew=2.0, color="black", zorder=7)
ax.annotate("nadir", (0, 0), textcoords="offset points", xytext=(9, 5), fontsize=9)
ax.set_aspect("equal")
ax.set_xlim(-21, 21)
ax.set_ylim(-21, 21)
ax.set_xlabel("Along_Track_Angle (deg)")
ax.set_ylabel("Cross_Track_Angle (deg)")
ax.set_title("Clock is this point's azimuth about\nthe origin, where it is undefined", fontsize=10)
ax.legend(loc="lower right", fontsize=8)
plt.colorbar(colours, ax=ax, shrink=0.85, pad=0.05, label="Clock_Angle (deg)")

ax = fig.add_subplot(grid[0, 2])
crossing = int(np.nanargmin(cone))
window = np.arange(max(crossing - 70, 0), min(crossing + 71, len(cone)))
ax.plot(seconds[window], clock_deg[window], lw=1.6, color="tab:purple", label="Clock_Angle")
ax.fill_between(seconds[window], 0, 360, where=cone[window] < gate, color="grey", alpha=0.25,
                label="inside the gate")
ax.set_ylim(0, 360)
ax.set_ylabel("Clock_Angle (deg)")
ax.set_xlabel("seconds from granule start")
ax.legend(loc="center left", fontsize=8)
rate_axis = ax.twinx()
rate_axis.plot(seconds[window], np.abs(np.gradient(clock_deg[window], seconds[window])), lw=1.1,
               color="crimson", label="|d(Clock_Angle)/dt|, ungated")
rate_axis.axhline(float(ds["Clock_Angle_Rate"].attrs["valid_range"][1]), color="crimson", lw=1.0, ls=":")
rate_axis.set_yscale("log")
rate_axis.set_ylabel("deg / s, log scale", color="crimson")
rate_axis.legend(loc="lower right", fontsize=8)
ax.set_title("One nadir crossing: %.0f deg of clock in %.1f s" %
             (abs(clock_deg[window][-1] - clock_deg[window][0]), seconds[window][-1] - seconds[window][0]),
             fontsize=10)

plt.show()

print("closest approach to nadir in this granule : %.3f deg" % np.nanmin(cone))
print("Clock_Angle from the orbital frame        : atan2(LOS . cross, LOS . in-track), max |diff| %.4f deg"
      % np.nanmax(np.abs(((np.degrees(np.arctan2(
          (line_of_sight * cross_axis).sum(1), (line_of_sight * in_track).sum(1))) % 360.0)
          - clock_deg + 180.0) % 360.0 - 180.0)))
print("peak ungated rate near the crossing       : %.0f deg/s against a valid_range of %s"
      % (np.nanmax(np.abs(np.gradient(clock_deg[window], seconds[window]))),
         ds["Clock_Angle_Rate"].attrs["valid_range"]))

In [ ]:
clock = ds["Clock_Angle"].values.astype("float64")
cone_rate = ds["Cone_Angle_Rate"].values.astype("float64")
clock_rate = ds["Clock_Angle_Rate"].values.astype("float64")

fig = plt.figure(figsize=(13, 5.5))

ax_polar = fig.add_subplot(1, 2, 1, projection="polar")
ax_polar.scatter(np.radians(clock), cone, c=seconds, s=3, cmap="plasma")
ax_polar.set_theta_zero_location("N")
ax_polar.set_theta_direction(-1)
ax_polar.set_rlim(0, 90)
ax_polar.set_title("Clock angle (azimuth) vs cone angle (radius)", pad=18)

ax_rate = fig.add_subplot(1, 2, 2)
ax_rate.plot(seconds, cone_rate, lw=0.8, label="Cone_Angle_Rate")
ax_rate.plot(seconds, clock_rate, lw=0.8, label="Clock_Angle_Rate")
ax_rate.fill_between(seconds, -80, 80, where=cone < gate, color="grey", alpha=0.3,
                     label=f"cone < {gate:g} deg (rate filled)")
declared_range = ds["Clock_Angle_Rate"].attrs.get("valid_range")
for edge in (float(declared_range[0]), float(declared_range[1])):
    ax_rate.axhline(edge, color="crimson", lw=1.0, ls=":")
ax_rate.plot([], [], color="crimson", lw=1.0, ls=":", label="Clock_Angle_Rate valid_range")
ax_rate.set_ylim(-80, 80)
ax_rate.set_xlabel("seconds from granule start")
ax_rate.set_ylabel("degrees / second")
ax_rate.set_title("Angle rates; shaded band is the near-nadir gate")
ax_rate.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

print("Clock_Angle range        : %.2f .. %.2f deg" % (np.nanmin(clock), np.nanmax(clock)))
print("samples inside the gate  : %d" % int(np.nansum(cone < gate)))
print("Clock_Angle_Rate NaN     : %d" % int(np.isnan(clock_rate).sum()))
print("Clock_Angle_Rate extremes: %.2f .. %.2f deg/s" % (np.nanmin(clock_rate), np.nanmax(clock_rate)))
print("declared valid_range     :", ds["Clock_Angle_Rate"].attrs.get("valid_range"))

### 1.8 Spacecraft state

`Satellite_Position` and `Satellite_Velocity` are J2000 (inertial), while the attitude quaternion
targets the Earth-fixed frame - a deliberate asymmetry in the product definition, and a common
source of confusion when combining them. `Satellite_Position_Start_Of_Hour` lives on a different
axis entirely: 24 epochs at the top of each UTC hour of the granule's day, not the sample grid.

The frame split is easier to see than to read. On the left, the J2000 fields: the 24 start-of-hour
positions ring the orbit, the in-granule position is one dot on that ring, and the velocity vector is
drawn scaled by ten minutes of flight. Those 24 epochs are one per UTC hour, not a trajectory - a
101-minute orbit steps past them, so consecutive epochs land at scattered places around the ring and
interpolating between them would be meaningless.

On the right, the attitude quaternion, rotated into the Earth-fixed frame it targets and drawn at the
spacecraft. Body `+Z` is the nadir-pointing axis and lands on the ellipsoid normal to a thousandth of
a degree. Body `+X` looks about three degrees off the ground-track tangent, and that gap is not a
pointing error: it is Earth rotation. The Earth-fixed velocity differs from the inertial one by
`omega x r`, roughly 0.43 km/s eastward here against 7.4 km/s of orbital motion, and once that is
added back `+X` lands on the inertial velocity to under half a degree. So the attitude is referenced
to the same in-track direction the cone, clock and look angles use, even though the quaternion itself
targets an Earth-fixed frame.

In [ ]:
hourly_position = ds["Satellite_Position_Start_Of_Hour"].values.astype("float64")
hourly_velocity = ds["Satellite_Velocity_Start_Of_Hour"].values.astype("float64")
attitude = np.stack([ds[f"Satellite_Attitude_Q{i}"].values for i in range(4)], axis=1).astype("float64")

orbit_normal = unit(np.cross(hourly_position, hourly_velocity))
precession = np.degrees(np.arccos(np.clip(orbit_normal @ orbit_normal[0], -1.0, 1.0)))
body_axes = quaternion_matrix(attitude)
state = len(satellite) // 2

fig = plt.figure(figsize=(14, 6.6))

ax = axes3d(fig, 1, 2, 1)
lon_mesh, lat_mesh = np.linspace(0.0, 2 * np.pi, 90), np.linspace(0.0, np.pi, 45)
ax.plot_surface(A_EARTH * np.outer(np.cos(lon_mesh), np.sin(lat_mesh)),
                A_EARTH * np.outer(np.sin(lon_mesh), np.sin(lat_mesh)),
                B_EARTH * np.outer(np.ones_like(lon_mesh), np.cos(lat_mesh)),
                color="#dce6f0", shade=False, edgecolor="none", antialiased=False, zorder=1)
for axis_index, name in enumerate(("J2000 X", "J2000 Y", "J2000 Z")):
    direction = np.eye(3)[axis_index]
    ax.plot(*np.stack([direction * A_EARTH, direction * 9300], axis=1), color="dimgrey", lw=1.0, ls="--", zorder=6)
    label3d(ax, direction * 9800, name, color="dimgrey", fontsize=8.5, zorder=11)
viewpoint_j2000 = unit(np.cross(orbit_normal[0], np.array([0.0, 0.0, 1.0])) + 0.55 * orbit_normal[0]
                       + 0.35 * np.array([0.0, 0.0, 1.0]))
in_plane = unit(hourly_position[0] - (hourly_position[0] @ orbit_normal[0]) * orbit_normal[0])
sweep = np.linspace(0.0, 2 * np.pi, 400)
ring = np.linalg.norm(hourly_position, axis=1).mean() * (
    np.cos(sweep)[:, None] * in_plane + np.sin(sweep)[:, None] * np.cross(orbit_normal[0], in_plane)
)
ax.plot(ring[:, 0], ring[:, 1], ring[:, 2], color="grey", lw=0.8, ls=":", zorder=5, label="orbit plane at hour 0")
in_front = (hourly_position @ viewpoint_j2000) > 0
for visible, alpha, label in ((in_front, 1.0, "Satellite_Position_Start_Of_Hour"),
                              (~in_front, 0.30, None)):
    ax.scatter(hourly_position[visible, 0], hourly_position[visible, 1], hourly_position[visible, 2],
               color="tab:blue", s=28, alpha=alpha, zorder=8, label=label)
ax.scatter(*position_j2000[state], color="black", s=65, zorder=10, label="Satellite_Position, in granule")
ax.plot(*np.stack([position_j2000[state], position_j2000[state] + velocity_j2000[state] * 600.0], axis=1),
        color="crimson", lw=2.4, zorder=9, label="Satellite_Velocity x 600 s")
frame3d(ax, viewpoint_j2000, 9900, zoom=1.42)
ax.legend(loc="upper left", fontsize=8)
ax.set_title("Position and velocity are J2000; the hourly grid spans a whole UTC day", fontsize=10)

ax = axes3d(fig, 1, 2, 2)
OMEGA_EARTH = np.array([0.0, 0.0, 7.292115e-5])   # Earth rotation vector, rad/s

ecef_nadir = -geodetic_normal(sub_lat, sub_lon)
velocity_ecef = np.gradient(satellite, seconds, axis=0)
# The same inertial velocity, resolved on Earth-fixed axes, so it can be compared with the body frame.
velocity_inertial = velocity_ecef + np.cross(OMEGA_EARTH, satellite)


def horizontal(vectors):
    """Drop the nadir component, leaving the direction along the local horizontal."""
    return unit(vectors - (vectors * ecef_nadir).sum(-1)[:, None] * ecef_nadir)


tangent = horizontal(velocity_ecef)
in_track_ecef = horizontal(velocity_inertial)
viewpoint = unit(-ecef_nadir[state] * np.cos(np.radians(76)) + tangent[state] * np.sin(np.radians(76)))
draw_globe(ax, viewpoint, coast_lw=0.7, graticule_step=10.0)
arm = 330.0
for axis_index, (colour, name) in enumerate((("tab:red", "body +X"), ("tab:green", "body +Y"),
                                            ("tab:blue", "body +Z"))):
    direction = body_axes[state][:, axis_index]
    ax.plot(*np.stack([satellite[state], satellite[state] + direction * arm], axis=1), color=colour, lw=2.8,
            zorder=9)
    label3d(ax, satellite[state] + direction * arm * 1.16, name, color=colour, fontsize=9, zorder=12, **HALO)
for direction, name in ((ecef_nadir[state], "geodetic nadir"), (tangent[state], "ground-track tangent")):
    ax.plot(*np.stack([satellite[state], satellite[state] + direction * arm * 1.95], axis=1), color="black",
            lw=1.1, ls="--", zorder=8)
    label3d(ax, satellite[state] + direction * arm * 2.18, name, fontsize=8.5, zorder=12, **HALO)
ax.scatter(*satellite[state], color="black", s=65, zorder=12)
ax.text2D(0.0, 0.03,
          "body +Z to geodetic nadir          %7.4f deg\n"
          "body +Z to geocentric nadir        %7.4f deg\n"
          "body +X to ground-track tangent    %7.4f deg\n"
          "body +X to inertial velocity       %7.4f deg"
          % (angle_between(body_axes[state][:, 2], ecef_nadir[state]),
             angle_between(body_axes[state][:, 2], -satellite[state]),
             angle_between(body_axes[state][:, 0], tangent[state]),
             angle_between(body_axes[state][:, 0], in_track_ecef[state])),
          transform=ax.transAxes, fontsize=8, family="monospace", va="bottom")
frame3d(ax, viewpoint, 640, center=tuple(satellite[state] + ecef_nadir[state] * 250.0), zoom=1.4)
ax.set_title("The attitude quaternion is Earth-fixed: body +Z is nadir", fontsize=10)

plt.tight_layout()
plt.show()

print("orbit plane drift over the 24 hourly epochs : %.3f deg" % precession.max())
print("  a sun-synchronous orbit precesses about 0.986 deg per day, so that is the expected signature")
print("consecutive hourly epochs are               : %.2f revolutions apart"
      % (3600.0 / (2 * np.pi * np.linalg.norm(hourly_position[0]) / np.linalg.norm(hourly_velocity[0]))))
print("body +Z against geodetic nadir, all samples  : max %.4f deg"
      % angle_between(body_axes[:, :, 2], ecef_nadir).max())
print("body +X against the ground-track tangent     : %.3f .. %.3f deg"
      % (angle_between(body_axes[:, :, 0], tangent).min(), angle_between(body_axes[:, :, 0], tangent).max()))
print("body +X against the inertial velocity        : %.3f .. %.3f deg"
      % (angle_between(body_axes[:, :, 0], in_track_ecef).min(),
         angle_between(body_axes[:, :, 0], in_track_ecef).max()))
print("Earth-fixed and inertial in-track directions : %.3f .. %.3f deg apart"
      % (angle_between(tangent, in_track_ecef).min(), angle_between(tangent, in_track_ecef).max()))

In [ ]:
position = ds["Satellite_Position"].values
velocity = ds["Satellite_Velocity"].values
quaternion = np.stack([ds[f"Satellite_Attitude_Q{i}"].values for i in range(4)], axis=1)
hourly_position = ds["Satellite_Position_Start_Of_Hour"].values

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for i, comp in enumerate("XYZ"):
    axes[0, 0].plot(seconds, position[:, i], lw=1.0, label=comp)
axes[0, 0].set_title("Satellite_Position (J2000)")
axes[0, 0].set_ylabel("km")

for i, comp in enumerate("XYZ"):
    axes[0, 1].plot(seconds, velocity[:, i], lw=1.0, label="V" + comp)
axes[0, 1].set_title("Satellite_Velocity (J2000)")
axes[0, 1].set_ylabel("km/s")

for i in range(4):
    axes[1, 0].plot(seconds, quaternion[:, i], lw=1.0, label=f"Q{i}")
axes[1, 0].set_title("Satellite_Attitude quaternion (Earth-fixed target frame)")
axes[1, 0].set_xlabel("seconds from granule start")

axes[1, 1].plot(np.arange(24), np.linalg.norm(hourly_position, axis=1), "o-", lw=1.0,
                label="|Position_Start_Of_Hour|")
axes[1, 1].axhline(np.linalg.norm(position, axis=1).mean(), color="tab:red", ls="--",
                   label="mean in-granule radius")
axes[1, 1].set_title("Start-of-hour grid: 24 epochs, not the sample grid")
axes[1, 1].set_xlabel("hour of the granule's UTC day")
axes[1, 1].set_ylabel("km")

for ax in axes.flat:
    ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()

print("orbital radius   : %.3f .. %.3f km" % (np.linalg.norm(position, axis=1).min(), np.linalg.norm(position, axis=1).max()))
print("speed            : %.4f .. %.4f km/s" % (np.linalg.norm(velocity, axis=1).min(), np.linalg.norm(velocity, axis=1).max()))
print("start-of-hour rows: %s, finite rows: %d" % (hourly_position.shape, int(np.isfinite(hourly_position).all(axis=1).sum())))

### 1.9 The scan in motion

Every plot so far has been a still. The fields are a function of pointing, though, and pointing is
what moves - so the last two figures in this half animate a five-second window covering two
limb-to-limb traversals, with the field values tracked alongside.

The first is the orbital frame of section 1.4 and 1.7 with the boresight actually sweeping, the cone
and clock arcs redrawn each frame, and the three fields that depend on them tracked to the right. The
second is the ground view: the boresight ray walking across the Earth and laying down the
`Latitude`/`Longitude` trail, which stalls whenever the boresight passes the limb.

These are the only expensive cells in the notebook. Each player embeds one PNG per frame, so at the
settings below the pair costs a few seconds to build and a few megabytes inside a *saved, executed*
copy of this notebook - nothing in the committed file, which is stripped of outputs. Set `ANIMATE` to
`False` to skip them, or lower `FRAMES`.

In [ ]:
from IPython.display import HTML, display
from matplotlib import animation

ANIMATE = True          # False skips both players
FRAMES = 44             # PNG frames per player; the dominant cost, linear in this number
FPS = 12
ANIMATION_DPI = 72

# Two traversals, so the boresight crosses nadir twice and the limb four times.
window = np.arange(turns[0], turns[2] + 1)
steps = window[np.linspace(0, len(window) - 1, FRAMES).astype(int)]
print("animating %.2f s of the granule in %d frames" % (seconds[steps[-1]] - seconds[steps[0]], FRAMES))


def show_animation(anim, figure):
    """Embed an animation as a self-contained player and drop the static figure behind it."""
    player = HTML(anim.to_jshtml(default_mode="loop"))
    plt.close(figure)
    display(player)
    print("embedded player: %.1f MB" % (len(player.data) / 1e6))

In [ ]:
if not ANIMATE:
    print("ANIMATE is False - skipping the orbital-frame player")
else:
    tracked = [
        ("Cone_Angle", cone, "tab:green"),
        ("Clock_Angle", clock_deg, "tab:purple"),
        ("Viewing_Zenith", view_zenith, "tab:blue"),
    ]

    fig = plt.figure(figsize=(12.5, 4.8), dpi=ANIMATION_DPI)
    layout = fig.add_gridspec(1, 2, width_ratios=(1.0, 1.3), wspace=0.05)

    # Left: the orbital frame. The plot axes are the frame axes by construction, so the frame
    # rotating with the spacecraft costs nothing - only the boresight components change.
    scene = axes3d(fig, layout[0, 0])
    for vector, colour, name in ((-ZENITH, "tab:blue", "nadir"), (IN_TRACK, "tab:green", "in-track"),
                                 (RIGHT, "tab:purple", "cross-track")):
        scene.plot(*np.stack([np.zeros(3), vector], axis=1), color=colour, lw=2.0, zorder=6)
        label3d(scene, vector * 1.16, name, color=colour, fontsize=8.5, zorder=11, **HALO)
    horizon_disc, horizon_angle = np.meshgrid(np.linspace(0.0, 0.95, 2), np.linspace(0.0, 2 * np.pi, 60))
    scene.plot_surface(horizon_disc * np.cos(horizon_angle), horizon_disc * np.sin(horizon_angle),
                       np.zeros_like(horizon_disc), color="#cfd9e4", alpha=0.35, shade=False,
                       edgecolor="none", zorder=1)
    boresight_line, = scene.plot([], [], [], color="crimson", lw=2.8, zorder=9)
    cone_arc, = scene.plot([], [], [], color="black", lw=1.4, zorder=10)
    clock_arc, = scene.plot([], [], [], color="tab:purple", lw=1.6, zorder=10)
    dropline, = scene.plot([], [], [], color="grey", lw=0.9, ls=":", zorder=8)
    scene.scatter(0, 0, 0, color="black", s=45, zorder=12)
    frame3d(scene, unit(0.85 * IN_TRACK + 0.55 * RIGHT + 0.40 * ZENITH), 1.12, zoom=1.75)
    readout = fig.text(0.03, 0.04, "", fontsize=9, family="monospace", va="bottom")

    # Right: the fields, with the window highlighted and a cursor riding along each trace.
    panels = layout[0, 1].subgridspec(len(tracked), 1, hspace=0.14)
    cursors, heads = [], []
    for row, (name, values, colour) in enumerate(tracked):
        axis = fig.add_subplot(panels[row, 0])
        axis.plot(seconds, values, lw=1.0, color=colour)
        axis.set_xlim(seconds[steps[0]], seconds[steps[-1]])
        axis.set_ylabel(name + " (deg)", fontsize=8)
        axis.tick_params(labelsize=8, labelbottom=row == len(tracked) - 1)
        if row == len(tracked) - 1:
            axis.set_xlabel("seconds from granule start", fontsize=9)
        cursors.append(axis.axvline(seconds[steps[0]], color="crimson", lw=1.1))
        heads.append(axis.plot([], [], marker="o", ms=6, color="crimson", zorder=6)[0])

    def draw_frame(index):
        sample = int(steps[index])
        components = unit(np.array([line_of_sight[sample] @ in_track[sample],
                                    line_of_sight[sample] @ cross_axis[sample],
                                    -line_of_sight[sample] @ nadir[sample]]))
        flat = np.array([components[0], components[1], 0.0])
        boresight_line.set_data_3d(*np.stack([np.zeros(3), components], axis=1))
        dropline.set_data_3d(*np.stack([components, flat], axis=1))
        cone_arc.set_data_3d(*arc_points(-ZENITH, components, 0.42).T)
        clock_arc.set_data_3d(*arc_points(IN_TRACK, flat, 0.66).T)
        zenith = view_zenith[sample]
        readout.set_text("t %5.2f s   cone %5.1f   clock %6.1f   VZA %9s"
                         % (seconds[sample], cone[sample], clock_deg[sample],
                            "%.1f" % zenith if np.isfinite(zenith) else "off Earth"))
        for cursor, head, (_, values, _) in zip(cursors, heads, tracked):
            cursor.set_xdata([seconds[sample], seconds[sample]])
            head.set_data([seconds[sample]], [values[sample]])
        return boresight_line, cone_arc, clock_arc, dropline, readout, *cursors, *heads

    show_animation(animation.FuncAnimation(fig, draw_frame, frames=len(steps),
                                           interval=int(1000 / FPS), blit=False), fig)

In [ ]:
if not ANIMATE:
    print("ANIMATE is False - skipping the ground-track player")
else:
    surface_track = geodetic_to_ecef(lat, lon)
    anchor = int(steps[len(steps) // 2])
    camera = unit(-nadir_ecef[anchor] * np.cos(np.radians(26)) - in_track_ecef[anchor] * np.sin(np.radians(26)))

    fig = plt.figure(figsize=(7.2, 6.3), dpi=ANIMATION_DPI)
    ax = axes3d(fig, 111)
    draw_globe(ax, camera)
    faint = near_side(surface_track[window] * 1.003, camera)
    ax.plot(faint[:, 0], faint[:, 1], faint[:, 2], color="grey", lw=0.8, alpha=0.6, zorder=5,
            label="where the boresight will go")
    ground = near_side(geodetic_to_ecef(sub_lat, sub_lon) * 1.004, camera)
    trail, = ax.plot([], [], [], color="tab:orange", lw=2.6, zorder=6, label="Latitude, Longitude so far")
    ax.plot(ground[:, 0], ground[:, 1], ground[:, 2], color="tab:blue", lw=2.4, ls=(0, (4, 2)), zorder=7,
            label="subsatellite track")
    ray, = ax.plot([], [], [], color="crimson", lw=2.2, zorder=9, label="Line_Of_Sight")
    head, = ax.plot([], [], [], marker="o", ms=7, color="crimson", zorder=10)
    ax.scatter(*satellite[anchor], color="black", s=45, zorder=11)
    frame3d(ax, camera, 6650, zoom=1.45)
    ax.legend(loc="upper left", fontsize=8)
    caption = fig.text(0.04, 0.035, "", fontsize=9, family="monospace", va="bottom")

    def draw_ground(index):
        sample = int(steps[index])
        drawn = window[(window <= sample) & np.isfinite(lat[window])]
        visible = near_side(surface_track[drawn] * 1.004, camera)
        trail.set_data_3d(visible[:, 0], visible[:, 1], visible[:, 2])
        if np.isfinite(lat[sample]):
            point = surface_track[sample]
            ray.set_data_3d(*np.stack([satellite[sample], point], axis=1))
            head.set_data_3d(*(point * 1.004)[:, None])
            caption.set_text("t %5.2f s   cone %5.1f deg   lat %7.2f   lon %8.2f"
                             % (seconds[sample], cone[sample], lat[sample], lon[sample]))
        else:
            empty = np.full(2, np.nan)
            ray.set_data_3d(empty, empty, empty)
            head.set_data_3d(empty, empty, empty)
            caption.set_text("t %5.2f s   cone %5.1f deg   boresight past the limb, no surface point"
                             % (seconds[sample], cone[sample]))
        return trail, ray, head, caption

    show_animation(animation.FuncAnimation(fig, draw_ground, frames=len(steps),
                                           interval=int(1000 / FPS), blit=False), fig)

## 2. Consistency checks

Each check below is an identity or physical relation that must hold if the fields agree with one
another. They are the first pass on a new granule: they need no external truth data, only the
product itself, so they catch frame mix-ups, unit errors, stale fields and broken fill logic.

The fields are stored as float32, so exact identities bottom out around `1e-5` degrees. Two checks
are approximate by nature and are labelled as such: the spherical sine rule ignores Earth's
flattening, and the rate comparisons re-derive a derivative by finite difference.

In [ ]:
def central_difference(values: np.ndarray, times: np.ndarray) -> np.ndarray:
    """Central finite difference, NaN wherever the three-point stencil touches a NaN."""
    out = np.full_like(values, np.nan)
    finite = np.isfinite(values)
    usable = finite[:-2] & finite[1:-1] & finite[2:]
    out[1:-1] = np.where(usable, (values[2:] - values[:-2]) / (times[2:] - times[:-2]), np.nan)
    return out


def angular_residual(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Absolute difference between two angle arrays in degrees, wrapped to [0, 180]."""
    diff = np.abs(a - b) % 360.0
    return np.minimum(diff, 360.0 - diff)


checks = []


def record(name, residual, tolerance, unit, n, note=""):
    """Append one check result; `residual` is compared against `tolerance`."""
    checks.append(
        {
            "check": name,
            "residual": residual,
            "tolerance": tolerance,
            "unit": unit,
            "n": n,
            "verdict": "pass" if (np.isfinite(residual) and residual <= tolerance) else "REVIEW",
            "note": note,
        }
    )


def finite_max(a, b):
    """Max absolute difference over samples where both inputs are finite, with the count."""
    both = np.isfinite(a) & np.isfinite(b)
    return (np.max(np.abs(a[both] - b[both])) if both.any() else np.nan), int(both.sum())

In [ ]:
# Definitional identities within the product.
for lat_name, colat_name in [
    ("Latitude", "Colatitude"),
    ("Subsatellite_Latitude", "Subsatellite_Colatitude"),
    ("Subsolar_Latitude", "Subsolar_Colatitude"),
]:
    residual, n = finite_max(90.0 - ds[lat_name].values.astype("float64"), ds[colat_name].values.astype("float64"))
    record(f"{colat_name} == 90 - {lat_name}", residual, 1e-4, "deg", n)

los = ds["Line_Of_Sight"].values.astype("float64")
norm = np.linalg.norm(los, axis=1)
record("|Line_Of_Sight| == 1", np.nanmax(np.abs(norm - 1)), 1e-6, "-", int(np.isfinite(norm).sum()))

quat_norm = np.linalg.norm(quaternion.astype("float64"), axis=1)
record("|Satellite_Attitude| == 1", np.nanmax(np.abs(quat_norm - 1)), 1e-6, "-", int(np.isfinite(quat_norm).sum()))

residual, n = finite_max(radius, np.linalg.norm(position.astype("float64"), axis=1))
record("Radius == |Satellite_Position|", residual, 1e-6, "km", n)

# Relative azimuth is defined from the other two azimuths.
predicted_raa = np.mod(vaz.astype("float64") - saz.astype("float64") + 180.0, 360.0)
both = np.isfinite(predicted_raa) & np.isfinite(raa)
record("Relative_Azimuth == mod(viewing - solar + 180, 360)",
       float(angular_residual(predicted_raa[both], raa[both].astype("float64")).max()), 1e-3, "deg", int(both.sum()))

# Cone angle is the resultant of the two look angles.
along = np.radians(ds["Along_Track_Angle"].values.astype("float64"))
cross = np.radians(ds["Cross_Track_Angle"].values.astype("float64"))
predicted_cone = np.degrees(np.arctan(np.hypot(np.tan(along), np.tan(cross))))
residual, n = finite_max(predicted_cone, cone)
record("Cone_Angle == atan(hypot(tan(along), tan(cross)))", residual, 1e-3, "deg", n)

# Solar zenith must equal the great-circle angle from the surface point to the subsolar point.
phi, lam = np.radians(lat.astype("float64")), np.radians(lon.astype("float64"))
sun_phi = np.radians(ds["Subsolar_Latitude"].values.astype("float64"))
sun_lam = np.radians(ds["Subsolar_Longitude"].values.astype("float64"))
cos_sza = np.sin(phi) * np.sin(sun_phi) + np.cos(phi) * np.cos(sun_phi) * np.cos(lam - sun_lam)
residual, n = finite_max(np.degrees(np.arccos(np.clip(cos_sza, -1, 1))), sza.astype("float64"))
record("Solar_Zenith == great-circle(surface, subsolar)", residual, 1e-2, "deg", n)

# Spherical sine rule links cone angle at the spacecraft to viewing zenith at the surface.
a_axis, b_axis = 6378.1366, 6356.7519
local_radius = np.sqrt(
    ((a_axis**2 * np.cos(phi)) ** 2 + (b_axis**2 * np.sin(phi)) ** 2)
    / ((a_axis * np.cos(phi)) ** 2 + (b_axis * np.sin(phi)) ** 2)
)
predicted_vza = np.degrees(np.arcsin(np.clip(radius / local_radius * np.sin(np.radians(cone)), -1, 1)))
residual, n = finite_max(predicted_vza, vza.astype("float64"))
record("Viewing_Zenith == asin(Rsc/Re * sin(cone))", residual, 0.5, "deg", n,
       "approximate: spherical sine rule ignores flattening")

# Rates must reproduce a finite difference of their own field.
residual, n = finite_max(central_difference(cone, seconds), cone_rate)
record("Cone_Angle_Rate == d(Cone_Angle)/dt", residual, 5e-2, "deg/s", n, "approximate: finite difference")

outside_gate = cone >= gate
clock_fd = central_difference(clock, seconds)
both = np.isfinite(clock_fd) & np.isfinite(clock_rate) & outside_gate
record("Clock_Angle_Rate == d(Clock_Angle)/dt outside the gate",
       float(np.max(np.abs(clock_fd[both] - clock_rate[both]))), 5e-2, "deg/s", int(both.sum()),
       "approximate: finite difference")

# Velocity must reproduce the derivative of position.
position_fd = np.stack([central_difference(position[:, i].astype("float64"), seconds) for i in range(3)], axis=1)
usable = np.isfinite(position_fd).all(axis=1)
record("Satellite_Velocity == d(Satellite_Position)/dt",
       float(np.max(np.abs(position_fd[usable] - velocity[usable].astype("float64")))), 1e-3, "km/s",
       int(usable.sum()), "approximate: finite difference")

# Fill logic: the gate must be applied exactly, and surface masks must agree exactly.
in_gate = np.isfinite(cone) & (cone < gate)
record("Clock_Angle_Rate filled exactly inside the gate",
       float(np.count_nonzero(np.isfinite(clock_rate) & in_gate)), 0, "samples", int(in_gate.sum()))
record("surface fields share one NaN mask",
       float(sum(not np.array_equal(np.isnan(ds[f].values), reference_mask) for f in SURFACE_FIELDS)),
       0, "fields", len(SURFACE_FIELDS))

# Motor telemetry against derived pointing: the elevation encoder and the cross-track look angle
# describe the same motion in oppositely wound frames, offset by the fixed boresight rotation.
elevation = ds["Elevation"].values.astype("float64")
cross_track = ds["Cross_Track_Angle"].values.astype("float64")
paired = np.isfinite(elevation) & np.isfinite(cross_track)
boresight_offset = float(np.median(cross_track[paired] + elevation[paired]))
record("Cross_Track_Angle == -Elevation + constant offset",
       float(np.max(np.abs(cross_track[paired] + elevation[paired] - boresight_offset))), 5e-3, "deg",
       int(paired.sum()), f"fixed offset {boresight_offset:+.4f} deg from the FK boresight rotation")

# Declared valid_range conformance across every variable that declares one.
violations = {}
for name in ds.data_vars:
    declared = ds[name].attrs.get("valid_range")
    if declared is None:
        continue
    values = ds[name].values.astype("float64")
    finite = np.isfinite(values)
    if not finite.any():
        continue
    outside = finite & ((values < float(declared[0])) | (values > float(declared[1])))
    if outside.any():
        violations[name] = (int(outside.sum()), float(np.nanmin(values[finite])), float(np.nanmax(values[finite])),
                            float(declared[0]), float(declared[1]))
record("all fields inside declared valid_range", float(len(violations)), 0, "fields", len(ds.data_vars))

results = pd.DataFrame(checks)
results["residual"] = results["residual"].map(lambda v: f"{v:.4g}")
results["tolerance"] = results["tolerance"].map(lambda v: f"{v:g}")
print(results.to_string(index=False))

if violations:
    print("\nvalid_range violations:")
    for name, (count, lo, hi, dlo, dhi) in violations.items():
        print(f"  {name}: {count} samples outside [{dlo:g}, {dhi:g}]; actual range [{lo:.4g}, {hi:.4g}]")

### 2.1 What the checks turn up

Every check passes on this granule. The definitional identities close at float32 precision and the
fill logic is exact: the near-nadir gate is applied to precisely the samples inside it, and all nine
surface fields share one NaN mask. The motor-encoder check is worth singling out because it is the
only one that leaves the curryer-derived fields entirely: `Cross_Track_Angle` reproduces the negated
`Elevation` encoder to about a thousandth of a degree once a constant offset is removed, so the CK,
the FK boresight rotation and the derived look angle all agree.

A clean sweep is not the interesting case, though. The `valid_range` check earns its place because it
recently failed: with a 6-degree gate, `Clock_Angle_Rate` reached about +/- 43 deg/s against a
declared range of `[-20, 20]`. The gate did suppress the true singularity - ungated the rate
approaches 8000 deg/s - but not by enough. Because the surviving maximum scales as one over the gate
squared, widening the gate to 12 degrees brings it to roughly 10 deg/s, at the cost of filling 15% of
samples rather than 7.5%. See `CLOCK_RATE_MIN_CONE_ANGLE_DEG` in `libera_rad/constants.py`; the width
remains a placeholder pending science confirmation (`TODO[LIBSDC-739]`), since other scan modes
change both the scan speed and how close the boresight passes to nadir.

That is the class of problem these checks exist to surface: nothing raised an exception, the product
wrote cleanly, and write-time conformance checking still passed, because `valid_range` is declarative
metadata rather than an enforced constraint.

## 3. Reproducing the fields from SPICE

The geometry fields come from curryer's `GeometryData`, wrapped by `libera_rad.geolocation`. Two
calls cover the product: one against the spacecraft observer for state and reference ground points,
one against the instrument observer for everything boresight-derived. `calculate_geometry` makes
both and joins them.

Recomputing on the product's own timestamps and differencing against the file is the strongest
single check available - it validates the whole chain from kernels through packaging.

In [ ]:
from libera_utils.libera_spice.kernel_manager import KernelManager

kernel_sources = [str(p) for p in sorted(KERNEL_DIR.iterdir()) if p.suffix in (".bc", ".bsp")]
for source in kernel_sources:
    print(Path(source).name)

# Kernels already in the libera_utils cache warn about a basename collision on every load, on both
# the warnings and logging channels. Harmless here - the cached file is a copy of this same file -
# and the message is long enough to bury the rest of the notebook, so both channels are quieted.
kernel_log = logging.getLogger("libera_utils.libera_spice.kernel_manager")
previous_level = kernel_log.level

kernel_manager = KernelManager()
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*basename conflict.*")
    kernel_log.setLevel(logging.ERROR)
    try:
        kernel_manager.load_libera_dynamic_kernels(kernel_sources, needs_naif_kernels=True, needs_static_kernels=True)
    finally:
        kernel_log.setLevel(previous_level)
kernel_manager.ensure_known_kernels_are_furnished()
print("\nkernels furnished")

In [ ]:
from libera_rad.geolocation import calculate_geometry, calculate_lat_lon_altitude

geometry = calculate_geometry(kernel_manager, time)
boresight_lla = calculate_lat_lon_altitude(kernel_manager, pd.DatetimeIndex(time))

print("curryer columns returned:", len(geometry.columns))
print(sorted(geometry.columns))

In [ ]:
# Product variable -> curryer column. Vector fields expand to three columns.
SCALAR_MAP = {
    "Subsatellite_Latitude": "subsatellite_latitude",
    "Subsatellite_Longitude": "subsatellite_longitude",
    "Subsatellite_Colatitude": "subsatellite_colatitude",
    "Subsolar_Latitude": "subsolar_latitude",
    "Subsolar_Longitude": "subsolar_longitude",
    "Subsolar_Colatitude": "subsolar_colatitude",
    "Radius_of_Satellite_from_Center_of_Earth": "spacecraft_radius",
    "Viewing_Zenith_Surface": "viewing_zenith",
    "Solar_Zenith_Surface": "solar_zenith",
    "Viewing_Azimuth_Surface_WRT_North": "viewing_azimuth",
    "Solar_Azimuth_Surface_WRT_North": "solar_azimuth",
    "Relative_Azimuth_Surface": "relative_azimuth",
    "Cone_Angle": "cone_angle",
    "Cone_Angle_Rate": "cone_angle_rate",
    "Clock_Angle": "clock_angle",
    "Along_Track_Angle": "along_track_angle",
    "Cross_Track_Angle": "cross_track_angle",
    "Satellite_Attitude_Q0": "attitude_q0",
    "Satellite_Attitude_Q1": "attitude_q1",
    "Satellite_Attitude_Q2": "attitude_q2",
    "Satellite_Attitude_Q3": "attitude_q3",
}
VECTOR_MAP = {
    "Satellite_Position": ["spacecraft_position_inertial_x", "spacecraft_position_inertial_y",
                           "spacecraft_position_inertial_z"],
    "Satellite_Velocity": ["spacecraft_velocity_inertial_x", "spacecraft_velocity_inertial_y",
                           "spacecraft_velocity_inertial_z"],
    "Line_Of_Sight": ["boresight_inertial_x", "boresight_inertial_y", "boresight_inertial_z"],
}

rows = []
for product_name, column in SCALAR_MAP.items():
    stored = ds[product_name].values.astype("float64")
    recomputed = geometry[column].to_numpy().astype("float64")
    residual, n = finite_max(stored, recomputed)
    rows.append({"product variable": product_name, "curryer column": column, "max |diff|": residual,
                 "n": n, "NaN masks match": bool(np.array_equal(np.isnan(stored), np.isnan(recomputed)))})

for product_name, columns in VECTOR_MAP.items():
    stored = ds[product_name].values.astype("float64")
    recomputed = geometry[columns].to_numpy().astype("float64")
    usable = np.isfinite(stored).all(axis=1) & np.isfinite(recomputed).all(axis=1)
    rows.append({"product variable": product_name, "curryer column": "(x, y, z)",
                 "max |diff|": float(np.max(np.abs(stored[usable] - recomputed[usable]))),
                 "n": int(usable.sum()),
                 "NaN masks match": bool(np.array_equal(np.isnan(stored).all(axis=1), np.isnan(recomputed).all(axis=1)))})

for product_name, column in [("Latitude", "lat"), ("Longitude", "lon"), ("Altitude", "alt")]:
    stored = ds[product_name].values.astype("float64")
    recomputed = boresight_lla[column].to_numpy().astype("float64")
    residual, n = finite_max(stored, recomputed)
    rows.append({"product variable": product_name, "curryer column": f"ellipsoid intersection [{column}]",
                 "max |diff|": residual, "n": n,
                 "NaN masks match": bool(np.array_equal(np.isnan(stored), np.isnan(recomputed)))})

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nlargest disagreement anywhere: %.3g" % comparison["max |diff|"].max())
print("NaN masks match everywhere   :", bool(comparison["NaN masks match"].all()))

Every field reproduces to float32 rounding and every NaN mask matches, so the product carries
exactly what the library computes for these timestamps and kernels.

`Clock_Angle_Rate` is absent from the table above because it is the one geometry field the product
modifies after curryer returns it. Applying the gate reproduces the stored field exactly, which
locates that decision unambiguously in `libera_rad` rather than in curryer.

In [ ]:
raw_clock_rate = geometry["clock_angle_rate"].to_numpy().astype("float64")
gated_clock_rate = np.where(geometry["cone_angle"].to_numpy() < gate, np.nan, raw_clock_rate)

residual, n = finite_max(clock_rate, gated_clock_rate)
print("max |product - gated curryer| : %.4g deg/s over %d samples" % (residual, n))
print("NaN masks match               :", np.array_equal(np.isnan(clock_rate), np.isnan(gated_clock_rate)))
print("ungated |rate| maximum        : %.1f deg/s" % np.nanmax(np.abs(raw_clock_rate)))
print("gated   |rate| maximum        : %.1f deg/s" % np.nanmax(np.abs(gated_clock_rate)))
print("declared valid_range          :", ds["Clock_Angle_Rate"].attrs.get("valid_range"))

### 3.1 An independent path to the subsatellite point

The checks in section 2 stayed inside the product. This one leaves it: rotate the stored J2000
`Satellite_Position` into the Earth-fixed frame with SPICE and convert to geodetic coordinates. The
result must land on `Subsatellite_Latitude`/`Longitude`, which curryer computed by a different
route. Agreement here exercises the inertial-to-Earth-fixed transform that any consumer combining
the position field with ground coordinates has to get right.

In [ ]:
from curryer import spicetime
from curryer import spicierpy as sp
from curryer.compute import spatial

ephemeris_time = spicetime.adapt(np.asarray(time, dtype="datetime64[ns]"), "dt64", "et")
sampled = np.arange(0, len(time), 100)

earth_fixed = np.array([sp.pxform("J2000", "ITRF93", float(et)) @ position[i]
                        for i, et in zip(sampled, ephemeris_time[sampled])])
geodetic = spatial.ecef_to_geodetic(earth_fixed, degrees=True, meters=False)

print("samples compared            : %d" % len(sampled))
print("max |lat - Subsatellite_Lat|: %.3g deg" % np.max(np.abs(geodetic[:, 1] - sub_lat[sampled])))
print("max |lon - Subsatellite_Lon|: %.3g deg" % np.max(np.abs(geodetic[:, 0] - sub_lon[sampled])))
print("geodetic altitude           : %.3f .. %.3f km" % (geodetic[:, 2].min(), geodetic[:, 2].max()))

### 3.2 The radiometer field of view

The product reports one lat/lon per sample: the boresight intersection. The instrument actually
integrates over a finite cone, defined in the Libera IK (`LIBERA_RAD`, circular, 1.0 degree
half-angle). Projecting the FOV boundary instead of just the boresight shows the real ground
footprint, which is what footprint matching against the WFOV camera has to work with.

The footprint is near-circular at nadir and stretches rapidly toward the limb, since the same angular
cone subtends far more ground at grazing incidence. The left panel places the footprints along the
scan; the right panel re-centres each one and plots it in kilometres, which is the only way to
compare shapes that sit thousands of kilometres apart.

In [ ]:
fov_shape, fov_frame, boresight_vector, _, boundary = sp.getfov(sp.obj.Body("LIBERA_RAD").id, 10)
half_angle = np.degrees(np.arccos(boundary[0][2] / np.linalg.norm(boundary[0])))
print("IK FOV: %s in %s, boresight %s, half-angle %.4f deg" % (fov_shape, fov_frame, boresight_vector, half_angle))


def fov_boundary_vectors(half_angle_deg: float, n: int = 121) -> np.ndarray:
    """Unit vectors around a circular FOV boundary, in the instrument frame."""
    azimuth = np.linspace(0, 2 * np.pi, n)
    theta = np.radians(half_angle_deg)
    return np.stack(
        [np.sin(theta) * np.cos(azimuth), np.sin(theta) * np.sin(azimuth), np.full_like(azimuth, np.cos(theta))],
        axis=1,
    )


# Sample a spread of cone angles that are all on the Earth, so every footprint closes.
candidates = np.where(np.isfinite(lat) & (cone < 60))[0]
chosen = candidates[np.argsort(cone[candidates])][np.linspace(0, len(candidates) - 1, 6).astype(int)]

chosen_ugps = spicetime.adapt(pd.DatetimeIndex(time[chosen]), "iso")
footprints, _, _ = spatial.compute_ellipsoid_intersection(
    chosen_ugps,
    sp.obj.Body("LIBERA_RAD", frame=True),
    custom_pointing_vectors=fov_boundary_vectors(half_angle),
    give_geodetic_output=True,
    give_lat_lon_in_degrees=True,
)

# groupby returns groups in uGPS order, which is not the cone-angle order of `chosen`;
# map each group back to its sample index explicitly rather than relying on position.
sample_of_ugps = dict(zip(chosen_ugps, chosen))

colors = dict(zip(chosen, plt.cm.viridis(np.linspace(0, 0.85, len(chosen)))))

fig = plt.figure(figsize=(14, 7))
ax_map = fig.add_subplot(1, 2, 1, projection=ccrs.Orthographic(float(np.nanmean(lon)), float(np.nanmean(lat))))
ax_map.set_global()
ax_map.coastlines(linewidth=0.6)
ax_map.gridlines(linewidth=0.3, linestyle="--", alpha=0.6)
ax_map.plot(lon[np.isfinite(lat)], lat[np.isfinite(lat)], color="grey", lw=0.6, alpha=0.7,
            transform=ccrs.PlateCarree(), zorder=5, label="boresight track")
ax_local = fig.add_subplot(1, 2, 2)

sizes = []
for ugps_key, group in footprints.groupby(level=0):
    index = sample_of_ugps[ugps_key]
    ring_lat, ring_lon = group["lat"].to_numpy(), group["lon"].to_numpy()
    if not np.isfinite(ring_lat).all():
        continue
    label = "cone %.1f deg" % cone[index]
    ax_map.plot(np.append(ring_lon, ring_lon[0]), np.append(ring_lat, ring_lat[0]), color=colors[index],
                lw=1.8, transform=ccrs.PlateCarree(), zorder=8)
    ax_map.scatter(ring_lon.mean(), ring_lat.mean(), color=colors[index], s=18,
                   transform=ccrs.PlateCarree(), zorder=9, label=label)

    # Re-centre each ring on its own centre and convert to kilometres, so shape and scale are
    # comparable between footprints that sit thousands of kilometres apart.
    centre_lat, centre_lon = ring_lat.mean(), ring_lon.mean()
    east_km = (ring_lon - centre_lon) * 111.32 * np.cos(np.radians(centre_lat))
    north_km = (ring_lat - centre_lat) * 111.32
    ax_local.plot(np.append(east_km, east_km[0]), np.append(north_km, north_km[0]),
                  color=colors[index], lw=1.8, label=label)
    sizes.append({"cone angle (deg)": round(float(cone[index]), 2),
                  "viewing zenith (deg)": round(float(vza[index]), 2),
                  "N-S extent (km)": round(float(north_km.max() - north_km.min()), 1),
                  "E-W extent (km)": round(float(east_km.max() - east_km.min()), 1)})

handles, labels = ax_map.get_legend_handles_labels()
order = sorted(range(len(labels)), key=lambda i: (labels[i] == "boresight track", labels[i]))
ax_map.legend([handles[i] for i in order], [labels[i] for i in order], loc="lower left", fontsize=9)
ax_map.set_title("Footprint locations along the scan")

ax_local.set_aspect("equal")
ax_local.axhline(0, color="grey", lw=0.5)
ax_local.axvline(0, color="grey", lw=0.5)
ax_local.set_xlabel("east offset from footprint centre (km)")
ax_local.set_ylabel("north offset (km)")
ax_local.set_title("Same footprints, each re-centred (km)")
local_handles, local_labels = ax_local.get_legend_handles_labels()
local_order = sorted(range(len(local_labels)), key=lambda i: local_labels[i])
ax_local.legend([local_handles[i] for i in local_order], [local_labels[i] for i in local_order],
                fontsize=9, loc="upper right")

fig.suptitle("Radiometer FOV footprint (%.1f deg half-angle) across the scan" % half_angle, fontsize=13)
plt.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

print(pd.DataFrame(sorted(sizes, key=lambda row: row["cone angle (deg)"])).to_string(index=False))

### 3.3 Any time grid

`calculate_geometry` takes arbitrary timestamps, so it is not limited to a product's sample grid.
The in-repo kernels cover a far longer window than the 30-second granule: the AZROT and ELSCAN CKs
run to 19:05:49. Recomputing at 10 Hz across the full window covers most of an orbit and shows the
along-track behaviour the short granule cannot.

This is also the pattern for computing geometry on any other input - a different granule, a
candidate kernel set, or a proposed scan pattern.

In [ ]:
import time as timing

# Window covered by the in-repo AZROT-CK / ELSCAN-CK for this granule.
long_grid = pd.date_range("2025-11-20T18:00:00", "2025-11-20T19:05:00", freq="100ms")

started = timing.perf_counter()
long_geometry = calculate_geometry(kernel_manager, long_grid.values)
elapsed = timing.perf_counter() - started
print("%d samples (%.1f min at 10 Hz) in %.1f s" % (len(long_grid), len(long_grid) / 600, elapsed))

long_seconds = (long_grid - long_grid[0]).total_seconds().to_numpy()
long_sub_lat = long_geometry["subsatellite_latitude"].to_numpy()
long_sub_lon = long_geometry["subsatellite_longitude"].to_numpy()
long_vza = long_geometry["viewing_zenith"].to_numpy()

fig = plt.figure(figsize=(14, 5.5))

ax_map = fig.add_subplot(1, 3, 1, projection=ccrs.Robinson())
ax_map.set_global()
ax_map.coastlines(linewidth=0.5)
ax_map.gridlines(linewidth=0.3, linestyle="--", alpha=0.5)
ax_map.scatter(long_sub_lon, long_sub_lat, s=1, c=long_seconds / 60, cmap="plasma",
               transform=ccrs.PlateCarree(), zorder=6)
ax_map.set_title("Subsatellite track,\nfull kernel window")

# The scan sawtooth has a ~5 s period, so plotting 39,000 samples of it fills a solid band. The
# orbit-scale panel shows the slow quantities; the zoom panel shows the scan still running.
ax_slow = fig.add_subplot(1, 3, 2)
ax_slow.plot(long_seconds / 60, long_sub_lat, lw=1.2, label="subsatellite latitude")

# Solar zenith at the boresight swings by tens of degrees within every 5 s sweep, so at orbit scale
# it is a band rather than a curve. The great-circle angle to the subsolar point evaluated at the
# subsatellite point is the smooth, scan-independent version of the same quantity.
sub_phi, sub_lam = np.radians(long_sub_lat), np.radians(long_sub_lon)
sun_phi_long = np.radians(long_geometry["subsolar_latitude"].to_numpy())
sun_lam_long = np.radians(long_geometry["subsolar_longitude"].to_numpy())
nadir_sza = np.degrees(np.arccos(np.clip(
    np.sin(sub_phi) * np.sin(sun_phi_long)
    + np.cos(sub_phi) * np.cos(sun_phi_long) * np.cos(sub_lam - sun_lam_long), -1, 1)))

ax_slow.plot(long_seconds / 60, long_geometry["solar_zenith"].to_numpy(), lw=0.5, color="tab:orange",
             alpha=0.25, label="solar zenith at boresight (scan spread)")
ax_slow.plot(long_seconds / 60, nadir_sza, lw=1.4, color="tab:red", label="solar zenith at nadir point")
ax_slow.axhline(90, color="grey", lw=0.8, ls="--", label="terminator (90 deg)")
ax_slow.set_xlabel("minutes from 18:00:00")
ax_slow.set_ylabel("degrees")
ax_slow.set_title("Orbit-scale geometry")
ax_slow.legend(loc="lower left", fontsize=8)

zoom = long_seconds <= 20
ax_zoom = fig.add_subplot(1, 3, 3)
ax_zoom.plot(long_seconds[zoom], long_geometry["cone_angle"].to_numpy()[zoom], lw=1.0, color="tab:green",
             label="cone_angle")
ax_zoom.set_xlabel("seconds from 18:00:00")
ax_zoom.set_ylabel("degrees")
ax_zoom.set_title("First 20 s: the scan\nis unchanged")
ax_zoom.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

print("subsatellite latitude : %.2f .. %.2f deg" % (long_sub_lat.min(), long_sub_lat.max()))
print("boresight on Earth    : %d of %d samples (%.1f%%)" % (
    np.isfinite(long_vza).sum(), len(long_vza), 100 * np.isfinite(long_vza).mean()))
print("cone angle            : %.2f .. %.2f deg" % (
    np.nanmin(long_geometry["cone_angle"]), np.nanmax(long_geometry["cone_angle"])))

### 3.4 Degraded modes

Two manifest configurations produce geometry without full pointing, and it is worth knowing what
they look like so a degraded granule is not mistaken for a broken one.

- **`jpss_only`** - no motor CK, so the instrument frame does not resolve and every boresight field
  is NaN. The spacecraft fields still come through, and geolocation falls back to the subsatellite
  point.
- **`use_geo: false`** - geolocation is bypassed entirely and every geometry field is written as its
  product fill value (`-999` for angles, `-9999` for distances). Note these are fill values, not
  NaN, so anything reading the file without `mask_and_scale` sees the sentinels directly.

In [ ]:
from libera_rad.geolocation import create_placeholder_geometry, subsatellite_lat_lon_alt

fallback = subsatellite_lat_lon_alt(geometry)
print("jpss_only geolocation falls back to the subsatellite point:")
print(fallback.head(3).to_string(), "\n")
print("max |fallback lat - Subsatellite_Latitude| : %.3g deg"
      % np.max(np.abs(fallback["lat"].to_numpy() - sub_lat)))
print("fallback altitude is the spacecraft altitude: %.3f .. %.3f km\n"
      % (fallback["alt"].min(), fallback["alt"].max()))

placeholder = create_placeholder_geometry(len(time))
distinct = {column: float(placeholder[column].iloc[0]) for column in placeholder.columns}
print("use_geo: false writes constant fill values across %d columns:" % len(placeholder.columns))
for value in sorted(set(distinct.values())):
    members = [name for name, fill in distinct.items() if fill == value]
    print("  %8.1f  <- %d columns (e.g. %s)" % (value, len(members), ", ".join(members[:3])))

## 4. Checklist for a new granule

Condensed from section 2. None of these need external truth data:

1. **Coverage.** Do the surface fields share one NaN mask? Is the cut-off at the geometric limb
   `asin(Re/Rsc)` rather than a contiguous block in time (which would indicate a kernel gap)?
2. **Definitional identities.** `Colatitude == 90 - Latitude`, `|Line_Of_Sight| == 1`,
   `|quaternion| == 1`, `Radius == |Satellite_Position|`.
3. **Angle closure.** `Relative_Azimuth == mod(viewing - solar + 180, 360)`;
   `Cone_Angle == atan(hypot(tan(along), tan(cross)))`.
4. **Independent geometry.** Does `Solar_Zenith_Surface` match the great-circle angle from the
   surface point to the subsolar point? Does the J2000 position, rotated to Earth-fixed, land on the
   subsatellite point?
5. **Motor telemetry vs derived pointing.** Does `Cross_Track_Angle` track the negated `Elevation`
   encoder to within a constant offset? This is the one check that spans the CK, the FK boresight
   rotation and the geometry fields at once.
6. **Rates.** Do `Cone_Angle_Rate` and `Clock_Angle_Rate` reproduce finite differences of their own
   fields, and does `Satellite_Velocity` reproduce the derivative of `Satellite_Position`?
7. **Fill logic.** Is `Clock_Angle_Rate` blanked on exactly the samples inside the near-nadir gate?
8. **Declared ranges.** Is every field inside its own `valid_range`? This is metadata, not an
   enforced constraint, so a conforming write proves nothing here.
9. **Reproducibility.** Does `calculate_geometry` on the product's own timestamps and kernels
   reproduce every field to float32 rounding, with matching NaN masks?

Where to look next in the code:

| Topic | Location |
| --- | --- |
| Field selection, observer split, gate | `libera_rad/geolocation.py`, `libera_rad/constants.py` |
| Packaging into product variables | `libera_rad/l1b.py` (`_package_l1b_product`) |
| Declared units, ranges and fills | `libera_rad/data/L1B_RAD-4CH_product_definition.yml` |
| Field definitions and conventions | `curryer.compute.geometry` module docstring |
| Kernels and frames | `libera_utils/data/spice/jpss4/` (FK, IK) |